<a href="https://colab.research.google.com/github/Fugant1/UMUSP-SemEval2026-Task9/blob/main/Tabelas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tabela 1

# Non-Gated XLM-RoBERTa 22L + T123 (Specific Loss)

## Unzip

In [ ]:
!unzip test_phase.zip

## Imports

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.49.0

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score
import numpy as np
import re
import os
import html
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    XLMRobertaModel,
    XLMRobertaPreTrainedModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from torch.utils.data import Dataset
import shutil
from google.colab import files, drive, runtime
import wandb

wandb.init(mode="disabled")

drive.mount('/content/drive')

## Dataset prep

In [ ]:
task_columns = ["polarization", "political", "racial/ethnic", "religious", "gender/sexual", "other", "stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]

task1_columns = ["polarization"]
task2_columns = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
task3_columns = ["stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]


In [ ]:
languages = [
    'arb', 'ben', 'mya', 'eng', 'hau', 'hin', 'nep', 'urd', 'zho', 'amh',
    'ori', 'fas', 'pol', 'pan', 'rus', 'spa', 'swa', 'tel', 'tur',
    'khm', 'deu', 'ita'
    ]
#the first and good result had no 'deu', 'ita', 'khm'
#'hau' is going bad too in the first training section

In [ ]:
def load_and_merge_robust(lang, split_name):
    """
    Loads Subtask 1 (Mandatory).
    If Subtask 2 or 3 are missing, fills their columns with 0.
    """
    # -------------------------------------------------------
    # A. Load Subtask 1 (The Anchor)
    # -------------------------------------------------------
    path_t1 = f'/content/subtask1/{split_name}/{lang}.csv'

    # If Task 1 is missing, we really can't do anything. Skip.
    if not os.path.exists(path_t1):
        if split_name == 'train': # Only warn for train, dev might be partial
            print(f"Skipping {lang.upper()}: No Subtask 1 file found.")
        return None

    # Load the anchor dataframe
    df = pd.read_csv(path_t1)

    # -------------------------------------------------------
    # B. Load Subtask 2 (Topic)
    # -------------------------------------------------------
    path_t2 = f'/content/subtask2/{split_name}/{lang}.csv'

    if os.path.exists(path_t2):
        df_t2 = pd.read_csv(path_t2)
        # LEFT MERGE: Keep all rows from Task 1. Match IDs from Task 2.
        df = df.merge(df_t2, on=['id', 'text'], how='left')
    else:
        # File missing? Fill columns with 0
        print(f"  -> {lang.upper()} missing Subtask 2. Filling columns with 0.")
        for col in task2_columns:
            df[col] = 0

    # -------------------------------------------------------
    # C. Load Subtask 3 (Critical Step for your missing files)
    # -------------------------------------------------------
    path_t3 = f'/content/subtask3/{split_name}/{lang}.csv'

    if os.path.exists(path_t3):
        df_t3 = pd.read_csv(path_t3)
        # LEFT MERGE again
        df = df.merge(df_t3, on=['id', 'text'], how='left')
    else:
        # This will trigger for ITA, MYA, POL, RUS
        print(f"  -> {lang.upper()} missing Subtask 3. Filling columns with 0.")
        for col in task3_columns:
            df[col] = 0

    # -------------------------------------------------------
    # D. Final Cleanup
    # -------------------------------------------------------
    # The merge might create NaNs if IDs didn't match perfectly. Fill them.
    df = df.fillna(0)

    # Tag the language
    df['language'] = lang

    return df

In [ ]:
df_train = None
df_val = None
train_frames = []
val_frames = []
for lang in languages:
    # Load Train
    df_train_lang = load_and_merge_robust(lang, 'train')
    if df_train_lang is not None:
        train_frames.append(df_train_lang)

    # Load Dev
    df_val_lang = load_and_merge_robust(lang, 'dev')
    if df_val_lang is not None:
        val_frames.append(df_val_lang)

In [ ]:
train = pd.concat(train_frames, ignore_index=True)
val = pd.concat(val_frames, ignore_index=True)

In [ ]:
train['text'] = train['text'].astype(str)
val['text'] = val['text'].astype(str)

train = train[train['text'].str.strip().str.len() > 0]
val = val[val['text'].str.strip().str.len() > 0]

print(f"Cleaned Train Shape: {train.shape}")
print(f"Cleaned Val Shape: {val.shape}")

In [ ]:
# Fix the dataset class by inheriting from torch.utils.data.Dataset
class PolarizationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length # Store max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding=False, max_length=self.max_length, return_tensors='pt')

        # Ensure consistent tensor conversion for all items
        item = {key: encoding[key].squeeze() for key in encoding.keys()}
        # CHANGE THIS LINE: Use torch.float instead of torch.long for multi-label classification
        item['labels'] = torch.tensor(label, dtype=torch.float)
        return item

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-large', num_labels=12)

# Create train and Test dataset for multilabel
train_dataset = PolarizationDataset(train['text'].tolist(), train[task_columns].values.tolist(), tokenizer)
val_dataset = PolarizationDataset(val['text'].tolist(), val[task_columns].values.tolist(), tokenizer)

## Model config

In [ ]:
class XLMRobertaForMultiLabelClassification(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)

        # 1. Gate (Polarization)
        self.polar_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1)
        )

        # 2. Topic Head
        self.topic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 5)
        )

        # 3. Toxic Head
        self.toxic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 6)
        )

        self.loss_polar_fct = nn.BCEWithLogitsLoss()
        self.loss_topic_fct = nn.BCEWithLogitsLoss()
        self.loss_toxic_fct = nn.BCEWithLogitsLoss()

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        # Forward Heads
        polar_logit = self.polar_classifier(pooler_output)
        topic_logits = self.topic_classifier(pooler_output)
        toxic_logits = self.toxic_classifier(pooler_output)

        all_logits = torch.cat((polar_logit, topic_logits, toxic_logits), dim=1)

        loss = None
        if labels is not None:
            loss_polar = self.loss_polar_fct(polar_logit, labels[:, 0].unsqueeze(1))
            loss_topic = self.loss_topic_fct(topic_logits, labels[:, 1:6])
            loss_toxic = self.loss_toxic_fct(toxic_logits, labels[:, 6:])

            loss = loss_polar + loss_topic + loss_toxic

        return {"loss": loss, "logits": all_logits}

In [ ]:
from transformers import XLMRobertaConfig

config = XLMRobertaConfig.from_pretrained("xlm-roberta-large")
config.num_labels = 12

config._experts_implementation = None
config._experts_implementation_internal = None

model = XLMRobertaForMultiLabelClassification.from_pretrained(
    "xlm-roberta-large",
    config=config
)

In [ ]:
import transformers
print(transformers.__version__)

## Training config

In [ ]:
# Define training arguments
training_args = TrainingArguments(
        num_train_epochs=6,              # Reduced from 10 (Critical)
        warmup_ratio=0.1,                # Slightly higher warmup for stability
        learning_rate=2e-5,            # Slightly lower peak LR for "Large" model
        weight_decay=0.01,

        per_device_train_batch_size=32,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=128,

        bf16=True,                       # Native A100 precision
        tf32=True,                       # TensorFloat-32 (Fast Matrix Math)
        optim="adamw_torch_fused",            # Faster compilation

        eval_strategy="steps",
        eval_steps = 200,
        save_strategy="steps",
        save_steps = 200,
        logging_steps=50,

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        dataloader_num_workers=8,
        dataloader_pin_memory=True,
        seed=190 # a primeira foi com seed = 42, segunda seed = 13, a terceira seed = 190
)

def compute_metrics_gated(p):
    logits = p.predictions
    labels = p.label_ids

    probs = 1 / (1 + np.exp(-logits))

    gate_probs = probs[:, 0]
    expert_probs = probs[:, 1:]

    gate_preds = (gate_probs > 0.5).astype(int)
    expert_preds_raw = (expert_probs > 0.5).astype(int)

    expert_preds_masked = expert_preds_raw * gate_preds[:, None]

    final_preds = np.column_stack((gate_preds, expert_preds_masked))

    # Definição dos índices das Tasks
    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]
    idx_t3 = [6, 7, 8, 9, 10, 11]

    # Cálculo dos F1 Scores Específicos
    f1_all = f1_score(labels, final_preds, average='macro')

    f1_polar = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro')

    # Task 2: Topics (Macro)
    f1_topics = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro')

    f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro')

    return {
        'f1_macro': f1_all,
        'f1_polar': f1_polar,
        'f1_topics': f1_topics,
        'f1_toxic': f1_toxic
    }

In [ ]:
def get_llrd_optimizer_parameters(model, learning_rate, weight_decay, lr_decay):
    # 1. Define distinct groups
    # The "Heads" get the Peak Learning Rate
    head_names = ["polar_classifier", "topic_classifier", "toxic_classifier"]

    # Standard exceptions (no weight decay for bias/LayerNorm)
    no_decay = ["bias", "LayerNorm.weight"]

    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
            "lr": learning_rate, # Peak LR (e.g., 2e-5)
        },
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": learning_rate,
        },
    ]

    # 2. Iterate Backwards through the XLM-R Encoder Layers
    # Start from top layer (closer to head) -> go down to embeddings
    layers = [model.roberta.embeddings] + list(model.roberta.encoder.layer)
    layers.reverse() # Reverse so layer 23 is first, embedding is last

    lr = learning_rate
    for layer in layers:
        lr *= lr_decay # Decay the LR (e.g., * 0.95)

        optimizer_grouped_parameters += [
            {
                "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
                "weight_decay": weight_decay,
                "lr": lr,
            },
            {
                "params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
                "lr": lr,
            },
        ]

    return optimizer_grouped_parameters

opt_parameters = get_llrd_optimizer_parameters(model, learning_rate=2e-5, weight_decay=0.01, lr_decay=0.95)
optimizer = torch.optim.AdamW(opt_parameters, lr=2e-5)

## Train

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_gated,  # Use the new metrics function
    data_collator=DataCollatorWithPadding(tokenizer),
    optimizers=(optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)]
)

In [ ]:
# Train the model
trainer.train()

## Evaluation

In [ ]:
results = trainer.evaluate()
print(f"Macro F1 score: {results['eval_f1_macro']}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import warnings

def evaluate_comprehensive(trainer, df_val, languages, task_columns, tokenizer, threshold=0.5):
    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]
    idx_t3 = [6, 7, 8, 9, 10, 11]

    results = []

    print("=" * 120)
    print(f"{'LANG':<5} | {'N':<5} | {'POLAR F1':<8} | {'TOPIC F1':<8} | {'ALL MACRO':<9} | {'GATE RATE':<9}")
    print("=" * 120)

    # 2. Loop per Language
    for lang in languages:
        df_lang = df_val[df_val['language'] == lang]
        if len(df_lang) == 0: continue

        lang_dataset = PolarizationDataset(
            df_lang['text'].tolist(),
            df_lang[task_columns].values.tolist(),
            tokenizer
        )

        output = trainer.predict(lang_dataset)

        probs = 1 / (1 + np.exp(-output.predictions))

        polar_preds = (probs[:, 0] > 0.5).astype(int)
        expert_preds_raw = (probs[:, 1:] > threshold).astype(int)

        expert_preds_masked = expert_preds_raw * polar_preds[:, None]
        final_preds = np.column_stack((polar_preds, expert_preds_masked))
        labels = output.label_ids

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            f1_polar = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro', zero_division=0)
            f1_topic = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro', zero_division=0)
            f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro', zero_division=0)
            f1_all = f1_score(labels, final_preds, average='macro', zero_division=0)

            polar_rate = np.mean(polar_preds)

        results.append({
            'language': lang,
            'samples': len(df_lang),
            'f1_polar': f1_polar,
            'f1_topic': f1_topic,
            'f1_toxic': f1_toxic,
            'f1_all': f1_all,
            'gate_open_rate': polar_rate
        })

        print(f"{lang.upper():<5} | {len(df_lang):<5} | {f1_polar:.4f}   | {f1_toxic:.4f}   | {f1_all:.4f}    | {polar_rate:.1%}")

    print("=" * 120)

    return pd.DataFrame(results).sort_values('f1_all', ascending=False)


detailed_results = evaluate_comprehensive(
    trainer,
    val,
    languages,
    task_columns,
    tokenizer,
    threshold=0.5
)

print("\n--- AVERAGES ---")
print(f"Polar:  {detailed_results['f1_polar'].mean():.4f}")
print(f"Topic: {detailed_results['f1_topic'].mean():.4f}")
print(f"Toxic: {detailed_results['f1_toxic'].mean():.4f}")

## Save

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [ ]:
drive_folder = "/content/drive/MyDrive/SemEvalMultiTaskResults/Tabelas/Tabela 4/NG-XLM-RoBERTa-22L-T123-SL_rerun_2"

if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)

In [ ]:
csv_path = os.path.join(drive_folder, "detailed_results.csv")
detailed_results.to_csv(csv_path, index=False)

In [ ]:
import time

In [ ]:
print(f"Saving model to {drive_folder}...")
trainer.save_model(drive_folder)
tokenizer.save_pretrained(drive_folder)
print("Model saved.")

print("Waiting 180 seconds for Drive background sync to finish...")
time.sleep(180)

if os.path.exists(os.path.join(drive_folder, "config.json")):
    print("SUCCESS: Config file verified. Safe to shutdown.")
    print("Shutting down runtime now, everything good")
    runtime.unassign()
else:
    print("WARNING: Something went wrong. File not found after waiting.")

print("Shutting down runtime now, no model saved :/")
runtime.unassign()

# Non-Gated XLM-RoBERTa 20L (no khm and ita) + T123 (Specific Loss)

## Unzip

In [ ]:
!unzip test_phase.zip

## Imports

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.49.0

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score
import numpy as np
import re
import os
import html
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    XLMRobertaModel,
    XLMRobertaPreTrainedModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from torch.utils.data import Dataset
import shutil
from google.colab import files, drive, runtime
import wandb

wandb.init(mode="disabled")

drive.mount('/content/drive')

## Dataset prep

In [ ]:
task_columns = ["polarization", "political", "racial/ethnic", "religious", "gender/sexual", "other", "stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]

task1_columns = ["polarization"]
task2_columns = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
task3_columns = ["stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]


In [ ]:
languages = [
    'arb', 'ben', 'mya', 'eng', 'hau', 'hin', 'nep', 'urd', 'zho', 'amh',
    'ori', 'fas', 'pol', 'pan', 'rus', 'spa', 'swa', 'tel', 'tur',
    'deu'
    ]
#the first and good result had no 'deu', 'ita', 'khm'
#'hau' is going bad too in the first training section

In [ ]:
def load_and_merge_robust(lang, split_name):
    """
    Loads Subtask 1 (Mandatory).
    If Subtask 2 or 3 are missing, fills their columns with 0.
    """
    # -------------------------------------------------------
    # A. Load Subtask 1 (The Anchor)
    # -------------------------------------------------------
    path_t1 = f'/content/subtask1/{split_name}/{lang}.csv'

    # If Task 1 is missing, we really can't do anything. Skip.
    if not os.path.exists(path_t1):
        if split_name == 'train': # Only warn for train, dev might be partial
            print(f"Skipping {lang.upper()}: No Subtask 1 file found.")
        return None

    # Load the anchor dataframe
    df = pd.read_csv(path_t1)

    # -------------------------------------------------------
    # B. Load Subtask 2 (Topic)
    # -------------------------------------------------------
    path_t2 = f'/content/subtask2/{split_name}/{lang}.csv'

    if os.path.exists(path_t2):
        df_t2 = pd.read_csv(path_t2)
        # LEFT MERGE: Keep all rows from Task 1. Match IDs from Task 2.
        df = df.merge(df_t2, on=['id', 'text'], how='left')
    else:
        # File missing? Fill columns with 0
        print(f"  -> {lang.upper()} missing Subtask 2. Filling columns with 0.")
        for col in task2_columns:
            df[col] = 0

    # -------------------------------------------------------
    # C. Load Subtask 3 (Critical Step for your missing files)
    # -------------------------------------------------------
    path_t3 = f'/content/subtask3/{split_name}/{lang}.csv'

    if os.path.exists(path_t3):
        df_t3 = pd.read_csv(path_t3)
        # LEFT MERGE again
        df = df.merge(df_t3, on=['id', 'text'], how='left')
    else:
        # This will trigger for ITA, MYA, POL, RUS
        print(f"  -> {lang.upper()} missing Subtask 3. Filling columns with 0.")
        for col in task3_columns:
            df[col] = 0

    # -------------------------------------------------------
    # D. Final Cleanup
    # -------------------------------------------------------
    # The merge might create NaNs if IDs didn't match perfectly. Fill them.
    df = df.fillna(0)

    # Tag the language
    df['language'] = lang

    return df

In [ ]:
df_train = None
df_val = None
train_frames = []
val_frames = []
for lang in languages:
    # Load Train
    df_train_lang = load_and_merge_robust(lang, 'train')
    if df_train_lang is not None:
        train_frames.append(df_train_lang)

    # Load Dev
    df_val_lang = load_and_merge_robust(lang, 'dev')
    if df_val_lang is not None:
        val_frames.append(df_val_lang)

In [ ]:
train = pd.concat(train_frames, ignore_index=True)
val = pd.concat(val_frames, ignore_index=True)

In [ ]:
train['text'] = train['text'].astype(str)
val['text'] = val['text'].astype(str)

train = train[train['text'].str.strip().str.len() > 0]
val = val[val['text'].str.strip().str.len() > 0]

print(f"Cleaned Train Shape: {train.shape}")
print(f"Cleaned Val Shape: {val.shape}")

In [ ]:
# Fix the dataset class by inheriting from torch.utils.data.Dataset
class PolarizationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length # Store max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding=False, max_length=self.max_length, return_tensors='pt')

        # Ensure consistent tensor conversion for all items
        item = {key: encoding[key].squeeze() for key in encoding.keys()}
        # CHANGE THIS LINE: Use torch.float instead of torch.long for multi-label classification
        item['labels'] = torch.tensor(label, dtype=torch.float)
        return item

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-large', num_labels=12)

# Create train and Test dataset for multilabel
train_dataset = PolarizationDataset(train['text'].tolist(), train[task_columns].values.tolist(), tokenizer)
val_dataset = PolarizationDataset(val['text'].tolist(), val[task_columns].values.tolist(), tokenizer)

## Model config

In [ ]:
class XLMRobertaForMultiLabelClassification(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)

        # 1. Gate (Polarization)
        self.polar_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1)
        )

        # 2. Topic Head
        self.topic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 5)
        )

        # 3. Toxic Head
        self.toxic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 6)
        )

        self.loss_polar_fct = nn.BCEWithLogitsLoss()
        self.loss_topic_fct = nn.BCEWithLogitsLoss()
        self.loss_toxic_fct = nn.BCEWithLogitsLoss()

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        # Forward Heads
        polar_logit = self.polar_classifier(pooler_output)
        topic_logits = self.topic_classifier(pooler_output)
        toxic_logits = self.toxic_classifier(pooler_output)

        all_logits = torch.cat((polar_logit, topic_logits, toxic_logits), dim=1)

        loss = None
        if labels is not None:
            loss_polar = self.loss_polar_fct(polar_logit, labels[:, 0].unsqueeze(1))
            loss_topic = self.loss_topic_fct(topic_logits, labels[:, 1:6])
            loss_toxic = self.loss_toxic_fct(toxic_logits, labels[:, 6:])

            loss = loss_polar + loss_topic + loss_toxic

        return {"loss": loss, "logits": all_logits}

In [ ]:
from transformers import XLMRobertaConfig

config = XLMRobertaConfig.from_pretrained("xlm-roberta-large")
config.num_labels = 12

config._experts_implementation = None
config._experts_implementation_internal = None

model = XLMRobertaForMultiLabelClassification.from_pretrained(
    "xlm-roberta-large",
    config=config
)

In [ ]:
import transformers
print(transformers.__version__)

## Training config

In [ ]:
# Define training arguments
training_args = TrainingArguments(
        num_train_epochs=6,              # Reduced from 10 (Critical)
        warmup_ratio=0.1,                # Slightly higher warmup for stability
        learning_rate=2e-5,            # Slightly lower peak LR for "Large" model
        weight_decay=0.01,

        per_device_train_batch_size=32,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=128,

        bf16=True,                       # Native A100 precision
        tf32=True,                       # TensorFloat-32 (Fast Matrix Math)
        optim="adamw_torch_fused",            # Faster compilation

        eval_strategy="steps",
        eval_steps = 200,
        save_strategy="steps",
        save_steps = 200,
        logging_steps=50,

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        dataloader_num_workers=8,
        dataloader_pin_memory=True,
        seed=42
)

def compute_metrics_gated(p):
    logits = p.predictions
    labels = p.label_ids

    probs = 1 / (1 + np.exp(-logits))

    gate_probs = probs[:, 0]
    expert_probs = probs[:, 1:]

    gate_preds = (gate_probs > 0.5).astype(int)
    expert_preds_raw = (expert_probs > 0.5).astype(int)

    expert_preds_masked = expert_preds_raw * gate_preds[:, None]

    final_preds = np.column_stack((gate_preds, expert_preds_masked))

    # Definição dos índices das Tasks
    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]
    idx_t3 = [6, 7, 8, 9, 10, 11]

    # Cálculo dos F1 Scores Específicos
    f1_all = f1_score(labels, final_preds, average='macro')

    f1_polar = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro')

    # Task 2: Topics (Macro)
    f1_topics = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro')

    f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro')

    return {
        'f1_macro': f1_all,
        'f1_polar': f1_polar,
        'f1_topics': f1_topics,
        'f1_toxic': f1_toxic
    }

In [ ]:
def get_llrd_optimizer_parameters(model, learning_rate, weight_decay, lr_decay):
    # 1. Define distinct groups
    # The "Heads" get the Peak Learning Rate
    head_names = ["polar_classifier", "topic_classifier", "toxic_classifier"]

    # Standard exceptions (no weight decay for bias/LayerNorm)
    no_decay = ["bias", "LayerNorm.weight"]

    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
            "lr": learning_rate, # Peak LR (e.g., 2e-5)
        },
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": learning_rate,
        },
    ]

    # 2. Iterate Backwards through the XLM-R Encoder Layers
    # Start from top layer (closer to head) -> go down to embeddings
    layers = [model.roberta.embeddings] + list(model.roberta.encoder.layer)
    layers.reverse() # Reverse so layer 23 is first, embedding is last

    lr = learning_rate
    for layer in layers:
        lr *= lr_decay # Decay the LR (e.g., * 0.95)

        optimizer_grouped_parameters += [
            {
                "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
                "weight_decay": weight_decay,
                "lr": lr,
            },
            {
                "params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
                "lr": lr,
            },
        ]

    return optimizer_grouped_parameters

opt_parameters = get_llrd_optimizer_parameters(model, learning_rate=2e-5, weight_decay=0.01, lr_decay=0.95)
optimizer = torch.optim.AdamW(opt_parameters, lr=2e-5)

## Train

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_gated,  # Use the new metrics function
    data_collator=DataCollatorWithPadding(tokenizer),
    optimizers=(optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)]
)

In [ ]:
# Train the model
trainer.train()

## Evaluation

In [ ]:
results = trainer.evaluate()
print(f"Macro F1 score: {results['eval_f1_macro']}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import warnings

def evaluate_comprehensive(trainer, df_val, languages, task_columns, tokenizer, threshold=0.5):
    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]
    idx_t3 = [6, 7, 8, 9, 10, 11]

    results = []

    print("=" * 120)
    print(f"{'LANG':<5} | {'N':<5} | {'POLAR F1':<8} | {'TOPIC F1':<8} | {'ALL MACRO':<9} | {'GATE RATE':<9}")
    print("=" * 120)

    # 2. Loop per Language
    for lang in languages:
        df_lang = df_val[df_val['language'] == lang]
        if len(df_lang) == 0: continue

        lang_dataset = PolarizationDataset(
            df_lang['text'].tolist(),
            df_lang[task_columns].values.tolist(),
            tokenizer
        )

        output = trainer.predict(lang_dataset)

        probs = 1 / (1 + np.exp(-output.predictions))

        polar_preds = (probs[:, 0] > 0.5).astype(int)
        expert_preds_raw = (probs[:, 1:] > threshold).astype(int)

        expert_preds_masked = expert_preds_raw * polar_preds[:, None]
        final_preds = np.column_stack((polar_preds, expert_preds_masked))
        labels = output.label_ids

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            f1_polar = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro', zero_division=0)
            f1_topic = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro', zero_division=0)
            f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro', zero_division=0)
            f1_all = f1_score(labels, final_preds, average='macro', zero_division=0)

            polar_rate = np.mean(polar_preds)

        results.append({
            'language': lang,
            'samples': len(df_lang),
            'f1_polar': f1_polar,
            'f1_topic': f1_topic,
            'f1_toxic': f1_toxic,
            'f1_all': f1_all,
            'gate_open_rate': polar_rate
        })

        print(f"{lang.upper():<5} | {len(df_lang):<5} | {f1_polar:.4f}   | {f1_toxic:.4f}   | {f1_all:.4f}    | {polar_rate:.1%}")

    print("=" * 120)

    return pd.DataFrame(results).sort_values('f1_all', ascending=False)


detailed_results = evaluate_comprehensive(
    trainer,
    val,
    languages,
    task_columns,
    tokenizer,
    threshold=0.5
)

print("\n--- AVERAGES ---")
print(f"Polar:  {detailed_results['f1_polar'].mean():.4f}")
print(f"Topic: {detailed_results['f1_topic'].mean():.4f}")
print(f"Toxic: {detailed_results['f1_toxic'].mean():.4f}")

## Save

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [ ]:
drive_folder = "/content/drive/MyDrive/SemEvalMultiTaskResults/Tabelas/Tabela 1/NG-XLM-RoBERTa-20Lnokhm-T123-SLv2"

if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)

In [ ]:
csv_path = os.path.join(drive_folder, "detailed_results.csv")
detailed_results.to_csv(csv_path, index=False)

In [ ]:
import time

In [ ]:
print(f"Saving model to {drive_folder}...")
trainer.save_model(drive_folder)
tokenizer.save_pretrained(drive_folder)
print("Model saved.")

print("Waiting 180 seconds for Drive background sync to finish...")
time.sleep(180)

if os.path.exists(os.path.join(drive_folder, "config.json")):
    print("SUCCESS: Config file verified. Safe to shutdown.")
    print("Shutting down runtime now, everything good")
    runtime.unassign()
else:
    print("WARNING: Something went wrong. File not found after waiting.")

print("Shutting down runtime now, no model saved :/")
runtime.unassign()

# Non-Gated XLM-RoBERTa 20L (no hau and ita) + T123 (Specific Loss)

## Unzip

In [ ]:
!unzip test_phase.zip

## Imports

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.49.0

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score
import numpy as np
import re
import os
import html
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    XLMRobertaModel,
    XLMRobertaPreTrainedModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from torch.utils.data import Dataset
import shutil
from google.colab import files, drive, runtime
import wandb

wandb.init(mode="disabled")

drive.mount('/content/drive')

## Dataset prep

In [ ]:
task_columns = ["polarization", "political", "racial/ethnic", "religious", "gender/sexual", "other", "stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]

task1_columns = ["polarization"]
task2_columns = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
task3_columns = ["stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]


In [ ]:
languages = [
    'arb', 'ben', 'mya', 'eng', 'khm', 'hin', 'nep', 'urd', 'zho', 'amh',
    'ori', 'fas', 'pol', 'pan', 'rus', 'spa', 'swa', 'tel', 'tur',
    'deu'
    ]
#the first and good result had no 'deu', 'ita', 'khm'
#'hau' is going bad too in the first training section

In [ ]:
def load_and_merge_robust(lang, split_name):
    """
    Loads Subtask 1 (Mandatory).
    If Subtask 2 or 3 are missing, fills their columns with 0.
    """
    # -------------------------------------------------------
    # A. Load Subtask 1 (The Anchor)
    # -------------------------------------------------------
    path_t1 = f'/content/subtask1/{split_name}/{lang}.csv'

    # If Task 1 is missing, we really can't do anything. Skip.
    if not os.path.exists(path_t1):
        if split_name == 'train': # Only warn for train, dev might be partial
            print(f"Skipping {lang.upper()}: No Subtask 1 file found.")
        return None

    # Load the anchor dataframe
    df = pd.read_csv(path_t1)

    # -------------------------------------------------------
    # B. Load Subtask 2 (Topic)
    # -------------------------------------------------------
    path_t2 = f'/content/subtask2/{split_name}/{lang}.csv'

    if os.path.exists(path_t2):
        df_t2 = pd.read_csv(path_t2)
        # LEFT MERGE: Keep all rows from Task 1. Match IDs from Task 2.
        df = df.merge(df_t2, on=['id', 'text'], how='left')
    else:
        # File missing? Fill columns with 0
        print(f"  -> {lang.upper()} missing Subtask 2. Filling columns with 0.")
        for col in task2_columns:
            df[col] = 0

    # -------------------------------------------------------
    # C. Load Subtask 3 (Critical Step for your missing files)
    # -------------------------------------------------------
    path_t3 = f'/content/subtask3/{split_name}/{lang}.csv'

    if os.path.exists(path_t3):
        df_t3 = pd.read_csv(path_t3)
        # LEFT MERGE again
        df = df.merge(df_t3, on=['id', 'text'], how='left')
    else:
        # This will trigger for ITA, MYA, POL, RUS
        print(f"  -> {lang.upper()} missing Subtask 3. Filling columns with 0.")
        for col in task3_columns:
            df[col] = 0

    # -------------------------------------------------------
    # D. Final Cleanup
    # -------------------------------------------------------
    # The merge might create NaNs if IDs didn't match perfectly. Fill them.
    df = df.fillna(0)

    # Tag the language
    df['language'] = lang

    return df

In [ ]:
df_train = None
df_val = None
train_frames = []
val_frames = []
for lang in languages:
    # Load Train
    df_train_lang = load_and_merge_robust(lang, 'train')
    if df_train_lang is not None:
        train_frames.append(df_train_lang)

    # Load Dev
    df_val_lang = load_and_merge_robust(lang, 'dev')
    if df_val_lang is not None:
        val_frames.append(df_val_lang)

In [ ]:
train = pd.concat(train_frames, ignore_index=True)
val = pd.concat(val_frames, ignore_index=True)

In [ ]:
train['text'] = train['text'].astype(str)
val['text'] = val['text'].astype(str)

train = train[train['text'].str.strip().str.len() > 0]
val = val[val['text'].str.strip().str.len() > 0]

print(f"Cleaned Train Shape: {train.shape}")
print(f"Cleaned Val Shape: {val.shape}")

In [ ]:
# Fix the dataset class by inheriting from torch.utils.data.Dataset
class PolarizationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length # Store max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding=False, max_length=self.max_length, return_tensors='pt')

        # Ensure consistent tensor conversion for all items
        item = {key: encoding[key].squeeze() for key in encoding.keys()}
        # CHANGE THIS LINE: Use torch.float instead of torch.long for multi-label classification
        item['labels'] = torch.tensor(label, dtype=torch.float)
        return item

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-large', num_labels=12)

# Create train and Test dataset for multilabel
train_dataset = PolarizationDataset(train['text'].tolist(), train[task_columns].values.tolist(), tokenizer)
val_dataset = PolarizationDataset(val['text'].tolist(), val[task_columns].values.tolist(), tokenizer)

## Model config

In [ ]:
class XLMRobertaForMultiLabelClassification(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)

        # 1. Gate (Polarization)
        self.polar_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1)
        )

        # 2. Topic Head
        self.topic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 5)
        )

        # 3. Toxic Head
        self.toxic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 6)
        )

        self.loss_polar_fct = nn.BCEWithLogitsLoss()
        self.loss_topic_fct = nn.BCEWithLogitsLoss()
        self.loss_toxic_fct = nn.BCEWithLogitsLoss()

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        # Forward Heads
        polar_logit = self.polar_classifier(pooler_output)
        topic_logits = self.topic_classifier(pooler_output)
        toxic_logits = self.toxic_classifier(pooler_output)

        all_logits = torch.cat((polar_logit, topic_logits, toxic_logits), dim=1)

        loss = None
        if labels is not None:
            loss_polar = self.loss_polar_fct(polar_logit, labels[:, 0].unsqueeze(1))
            loss_topic = self.loss_topic_fct(topic_logits, labels[:, 1:6])
            loss_toxic = self.loss_toxic_fct(toxic_logits, labels[:, 6:])

            loss = loss_polar + loss_topic + loss_toxic

        return {"loss": loss, "logits": all_logits}

In [ ]:
from transformers import XLMRobertaConfig

config = XLMRobertaConfig.from_pretrained("xlm-roberta-large")
config.num_labels = 12

config._experts_implementation = None
config._experts_implementation_internal = None

model = XLMRobertaForMultiLabelClassification.from_pretrained(
    "xlm-roberta-large",
    config=config
)

In [ ]:
import transformers
print(transformers.__version__)

## Training config

In [ ]:
# Define training arguments
training_args = TrainingArguments(
        num_train_epochs=6,              # Reduced from 10 (Critical)
        warmup_ratio=0.1,                # Slightly higher warmup for stability
        learning_rate=2e-5,            # Slightly lower peak LR for "Large" model
        weight_decay=0.01,

        per_device_train_batch_size=32,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=128,

        bf16=True,                       # Native A100 precision
        tf32=True,                       # TensorFloat-32 (Fast Matrix Math)
        optim="adamw_torch_fused",            # Faster compilation

        eval_strategy="steps",
        eval_steps = 200,
        save_strategy="steps",
        save_steps = 200,
        logging_steps=50,

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        dataloader_num_workers=8,
        dataloader_pin_memory=True,
        seed=42
)

def compute_metrics_gated(p):
    logits = p.predictions
    labels = p.label_ids

    probs = 1 / (1 + np.exp(-logits))

    gate_probs = probs[:, 0]
    expert_probs = probs[:, 1:]

    gate_preds = (gate_probs > 0.5).astype(int)
    expert_preds_raw = (expert_probs > 0.5).astype(int)

    expert_preds_masked = expert_preds_raw * gate_preds[:, None]

    final_preds = np.column_stack((gate_preds, expert_preds_masked))

    # Definição dos índices das Tasks
    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]
    idx_t3 = [6, 7, 8, 9, 10, 11]

    # Cálculo dos F1 Scores Específicos
    f1_all = f1_score(labels, final_preds, average='macro')

    f1_polar = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro')

    # Task 2: Topics (Macro)
    f1_topics = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro')

    f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro')

    return {
        'f1_macro': f1_all,
        'f1_polar': f1_polar,
        'f1_topics': f1_topics,
        'f1_toxic': f1_toxic
    }

In [ ]:
def get_llrd_optimizer_parameters(model, learning_rate, weight_decay, lr_decay):
    # 1. Define distinct groups
    # The "Heads" get the Peak Learning Rate
    head_names = ["polar_classifier", "topic_classifier", "toxic_classifier"]

    # Standard exceptions (no weight decay for bias/LayerNorm)
    no_decay = ["bias", "LayerNorm.weight"]

    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
            "lr": learning_rate, # Peak LR (e.g., 2e-5)
        },
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": learning_rate,
        },
    ]

    # 2. Iterate Backwards through the XLM-R Encoder Layers
    # Start from top layer (closer to head) -> go down to embeddings
    layers = [model.roberta.embeddings] + list(model.roberta.encoder.layer)
    layers.reverse() # Reverse so layer 23 is first, embedding is last

    lr = learning_rate
    for layer in layers:
        lr *= lr_decay # Decay the LR (e.g., * 0.95)

        optimizer_grouped_parameters += [
            {
                "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
                "weight_decay": weight_decay,
                "lr": lr,
            },
            {
                "params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
                "lr": lr,
            },
        ]

    return optimizer_grouped_parameters

opt_parameters = get_llrd_optimizer_parameters(model, learning_rate=2e-5, weight_decay=0.01, lr_decay=0.95)
optimizer = torch.optim.AdamW(opt_parameters, lr=2e-5)

## Train

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_gated,  # Use the new metrics function
    data_collator=DataCollatorWithPadding(tokenizer),
    optimizers=(optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)]
)

In [ ]:
# Train the model
trainer.train()

## Evaluation

In [ ]:
results = trainer.evaluate()
print(f"Macro F1 score: {results['eval_f1_macro']}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import warnings

def evaluate_comprehensive(trainer, df_val, languages, task_columns, tokenizer, threshold=0.5):
    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]
    idx_t3 = [6, 7, 8, 9, 10, 11]

    results = []

    print("=" * 120)
    print(f"{'LANG':<5} | {'N':<5} | {'POLAR F1':<8} | {'TOPIC F1':<8} | {'ALL MACRO':<9} | {'GATE RATE':<9}")
    print("=" * 120)

    # 2. Loop per Language
    for lang in languages:
        df_lang = df_val[df_val['language'] == lang]
        if len(df_lang) == 0: continue

        lang_dataset = PolarizationDataset(
            df_lang['text'].tolist(),
            df_lang[task_columns].values.tolist(),
            tokenizer
        )

        output = trainer.predict(lang_dataset)

        probs = 1 / (1 + np.exp(-output.predictions))

        polar_preds = (probs[:, 0] > 0.5).astype(int)
        expert_preds_raw = (probs[:, 1:] > threshold).astype(int)

        expert_preds_masked = expert_preds_raw * polar_preds[:, None]
        final_preds = np.column_stack((polar_preds, expert_preds_masked))
        labels = output.label_ids

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            f1_polar = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro', zero_division=0)
            f1_topic = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro', zero_division=0)
            f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro', zero_division=0)
            f1_all = f1_score(labels, final_preds, average='macro', zero_division=0)

            polar_rate = np.mean(polar_preds)

        results.append({
            'language': lang,
            'samples': len(df_lang),
            'f1_polar': f1_polar,
            'f1_topic': f1_topic,
            'f1_toxic': f1_toxic,
            'f1_all': f1_all,
            'gate_open_rate': polar_rate
        })

        print(f"{lang.upper():<5} | {len(df_lang):<5} | {f1_polar:.4f}   | {f1_toxic:.4f}   | {f1_all:.4f}    | {polar_rate:.1%}")

    print("=" * 120)

    return pd.DataFrame(results).sort_values('f1_all', ascending=False)


detailed_results = evaluate_comprehensive(
    trainer,
    val,
    languages,
    task_columns,
    tokenizer,
    threshold=0.5
)

print("\n--- AVERAGES ---")
print(f"Polar:  {detailed_results['f1_polar'].mean():.4f}")
print(f"Topic: {detailed_results['f1_topic'].mean():.4f}")
print(f"Toxic: {detailed_results['f1_toxic'].mean():.4f}")

## Save

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [ ]:
drive_folder = "/content/drive/MyDrive/SemEvalMultiTaskResults/Tabelas/Tabela 1/NG-XLM-RoBERTa-20Lnohau-T123-SL"

if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)

In [ ]:
csv_path = os.path.join(drive_folder, "detailed_results.csv")
detailed_results.to_csv(csv_path, index=False)

In [ ]:
import time

In [ ]:
print(f"Saving model to {drive_folder}...")
trainer.save_model(drive_folder)
tokenizer.save_pretrained(drive_folder)
print("Model saved.")

print("Waiting 180 seconds for Drive background sync to finish...")
time.sleep(180)

if os.path.exists(os.path.join(drive_folder, "config.json")):
    print("SUCCESS: Config file verified. Safe to shutdown.")
    print("Shutting down runtime now, everything good")
    runtime.unassign()
else:
    print("WARNING: Something went wrong. File not found after waiting.")

print("Shutting down runtime now, no model saved :/")
runtime.unassign()

# Non-Gated XLM-RoBERTa 19L + T123 (Specific Loss)

## Unzip

In [ ]:
!unzip test_phase.zip

## Imports

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.49.0

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score
import numpy as np
import re
import os
import html
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    XLMRobertaModel,
    XLMRobertaPreTrainedModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from torch.utils.data import Dataset
import shutil
from google.colab import files, drive, runtime
import wandb

wandb.init(mode="disabled")

drive.mount('/content/drive')

## Dataset prep

In [ ]:
task_columns = ["polarization", "political", "racial/ethnic", "religious", "gender/sexual", "other", "stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]

task1_columns = ["polarization"]
task2_columns = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
task3_columns = ["stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]


In [ ]:
languages = [
    'arb', 'ben', 'mya', 'eng', 'hau', 'hin', 'nep', 'urd', 'zho', 'amh',
    'ori', 'fas', 'pol', 'pan', 'rus', 'spa', 'swa', 'tel', 'tur',
    ]
#the first and good result had no 'deu', 'ita', 'khm'
#'hau' is going bad too in the first training section

In [ ]:
def load_and_merge_robust(lang, split_name):
    """
    Loads Subtask 1 (Mandatory).
    If Subtask 2 or 3 are missing, fills their columns with 0.
    """
    # -------------------------------------------------------
    # A. Load Subtask 1 (The Anchor)
    # -------------------------------------------------------
    path_t1 = f'/content/subtask1/{split_name}/{lang}.csv'

    # If Task 1 is missing, we really can't do anything. Skip.
    if not os.path.exists(path_t1):
        if split_name == 'train': # Only warn for train, dev might be partial
            print(f"Skipping {lang.upper()}: No Subtask 1 file found.")
        return None

    # Load the anchor dataframe
    df = pd.read_csv(path_t1)

    # -------------------------------------------------------
    # B. Load Subtask 2 (Topic)
    # -------------------------------------------------------
    path_t2 = f'/content/subtask2/{split_name}/{lang}.csv'

    if os.path.exists(path_t2):
        df_t2 = pd.read_csv(path_t2)
        # LEFT MERGE: Keep all rows from Task 1. Match IDs from Task 2.
        df = df.merge(df_t2, on=['id', 'text'], how='left')
    else:
        # File missing? Fill columns with 0
        print(f"  -> {lang.upper()} missing Subtask 2. Filling columns with 0.")
        for col in task2_columns:
            df[col] = 0

    # -------------------------------------------------------
    # C. Load Subtask 3 (Critical Step for your missing files)
    # -------------------------------------------------------
    path_t3 = f'/content/subtask3/{split_name}/{lang}.csv'

    if os.path.exists(path_t3):
        df_t3 = pd.read_csv(path_t3)
        # LEFT MERGE again
        df = df.merge(df_t3, on=['id', 'text'], how='left')
    else:
        # This will trigger for ITA, MYA, POL, RUS
        print(f"  -> {lang.upper()} missing Subtask 3. Filling columns with 0.")
        for col in task3_columns:
            df[col] = 0

    # -------------------------------------------------------
    # D. Final Cleanup
    # -------------------------------------------------------
    # The merge might create NaNs if IDs didn't match perfectly. Fill them.
    df = df.fillna(0)

    # Tag the language
    df['language'] = lang

    return df

In [ ]:
df_train = None
df_val = None
train_frames = []
val_frames = []
for lang in languages:
    # Load Train
    df_train_lang = load_and_merge_robust(lang, 'train')
    if df_train_lang is not None:
        train_frames.append(df_train_lang)

    # Load Dev
    df_val_lang = load_and_merge_robust(lang, 'dev')
    if df_val_lang is not None:
        val_frames.append(df_val_lang)

In [ ]:
train = pd.concat(train_frames, ignore_index=True)
val = pd.concat(val_frames, ignore_index=True)

In [ ]:
train['text'] = train['text'].astype(str)
val['text'] = val['text'].astype(str)

train = train[train['text'].str.strip().str.len() > 0]
val = val[val['text'].str.strip().str.len() > 0]

print(f"Cleaned Train Shape: {train.shape}")
print(f"Cleaned Val Shape: {val.shape}")

In [ ]:
# Fix the dataset class by inheriting from torch.utils.data.Dataset
class PolarizationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length # Store max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding=False, max_length=self.max_length, return_tensors='pt')

        # Ensure consistent tensor conversion for all items
        item = {key: encoding[key].squeeze() for key in encoding.keys()}
        # CHANGE THIS LINE: Use torch.float instead of torch.long for multi-label classification
        item['labels'] = torch.tensor(label, dtype=torch.float)
        return item

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-large', num_labels=12)

# Create train and Test dataset for multilabel
train_dataset = PolarizationDataset(train['text'].tolist(), train[task_columns].values.tolist(), tokenizer)
val_dataset = PolarizationDataset(val['text'].tolist(), val[task_columns].values.tolist(), tokenizer)

## Model config

In [ ]:
class XLMRobertaForMultiLabelClassification(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)

        # 1. Gate (Polarization)
        self.polar_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1)
        )

        # 2. Topic Head
        self.topic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 5)
        )

        # 3. Toxic Head
        self.toxic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 6)
        )

        self.loss_polar_fct = nn.BCEWithLogitsLoss()
        self.loss_topic_fct = nn.BCEWithLogitsLoss()
        self.loss_toxic_fct = nn.BCEWithLogitsLoss()

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        # Forward Heads
        polar_logit = self.polar_classifier(pooler_output)
        topic_logits = self.topic_classifier(pooler_output)
        toxic_logits = self.toxic_classifier(pooler_output)

        all_logits = torch.cat((polar_logit, topic_logits, toxic_logits), dim=1)

        loss = None
        if labels is not None:
            loss_polar = self.loss_polar_fct(polar_logit, labels[:, 0].unsqueeze(1))
            loss_topic = self.loss_topic_fct(topic_logits, labels[:, 1:6])
            loss_toxic = self.loss_toxic_fct(toxic_logits, labels[:, 6:])

            loss = loss_polar + loss_topic + loss_toxic

        return {"loss": loss, "logits": all_logits}

In [ ]:
# Load the model
model = XLMRobertaForMultiLabelClassification.from_pretrained('xlm-roberta-large', num_labels=12) # use 6 labels

## Training config

In [ ]:
# Define training arguments
training_args = TrainingArguments(
        num_train_epochs=6,              # Reduced from 10 (Critical)
        warmup_ratio=0.1,                # Slightly higher warmup for stability
        learning_rate=2e-5,            # Slightly lower peak LR for "Large" model
        weight_decay=0.01,

        per_device_train_batch_size=32,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=128,

        bf16=True,                       # Native A100 precision
        tf32=True,                       # TensorFloat-32 (Fast Matrix Math)
        optim="adamw_torch_fused",            # Faster compilation

        eval_strategy="steps",
        eval_steps = 200,
        save_strategy="steps",
        save_steps = 200,
        logging_steps=50,

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        dataloader_num_workers=8,
        dataloader_pin_memory=True,
        seed=42
)

def compute_metrics_gated(p):
    logits = p.predictions
    labels = p.label_ids

    probs = 1 / (1 + np.exp(-logits))

    gate_probs = probs[:, 0]
    expert_probs = probs[:, 1:]

    gate_preds = (gate_probs > 0.5).astype(int)
    expert_preds_raw = (expert_probs > 0.5).astype(int)

    expert_preds_masked = expert_preds_raw * gate_preds[:, None]

    final_preds = np.column_stack((gate_preds, expert_preds_masked))

    # Definição dos índices das Tasks
    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]
    idx_t3 = [6, 7, 8, 9, 10, 11]

    # Cálculo dos F1 Scores Específicos
    f1_all = f1_score(labels, final_preds, average='macro')

    f1_polar = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro')

    # Task 2: Topics (Macro)
    f1_topics = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro')

    f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro')

    return {
        'f1_macro': f1_all,
        'f1_polar': f1_polar,
        'f1_topics': f1_topics,
        'f1_toxic': f1_toxic
    }

In [ ]:
def get_llrd_optimizer_parameters(model, learning_rate, weight_decay, lr_decay):
    # 1. Define distinct groups
    # The "Heads" get the Peak Learning Rate
    head_names = ["polar_classifier", "topic_classifier", "toxic_classifier"]

    # Standard exceptions (no weight decay for bias/LayerNorm)
    no_decay = ["bias", "LayerNorm.weight"]

    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
            "lr": learning_rate, # Peak LR (e.g., 2e-5)
        },
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": learning_rate,
        },
    ]

    # 2. Iterate Backwards through the XLM-R Encoder Layers
    # Start from top layer (closer to head) -> go down to embeddings
    layers = [model.roberta.embeddings] + list(model.roberta.encoder.layer)
    layers.reverse() # Reverse so layer 23 is first, embedding is last

    lr = learning_rate
    for layer in layers:
        lr *= lr_decay # Decay the LR (e.g., * 0.95)

        optimizer_grouped_parameters += [
            {
                "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
                "weight_decay": weight_decay,
                "lr": lr,
            },
            {
                "params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
                "lr": lr,
            },
        ]

    return optimizer_grouped_parameters

opt_parameters = get_llrd_optimizer_parameters(model, learning_rate=2e-5, weight_decay=0.01, lr_decay=0.95)
optimizer = torch.optim.AdamW(opt_parameters, lr=2e-5)

## Train

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_gated,  # Use the new metrics function
    data_collator=DataCollatorWithPadding(tokenizer),
    optimizers=(optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)]
)

In [ ]:
# Train the model
trainer.train()

## Evaluation

In [ ]:
results = trainer.evaluate()
print(f"Macro F1 score: {results['eval_f1_macro']}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import warnings

def evaluate_comprehensive(trainer, df_val, languages, task_columns, tokenizer, threshold=0.5):
    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]
    idx_t3 = [6, 7, 8, 9, 10, 11]

    results = []

    print("=" * 120)
    print(f"{'LANG':<5} | {'N':<5} | {'POLAR F1':<8} | {'TOPIC F1':<8} | {'ALL MACRO':<9} | {'GATE RATE':<9}")
    print("=" * 120)

    # 2. Loop per Language
    for lang in languages:
        df_lang = df_val[df_val['language'] == lang]
        if len(df_lang) == 0: continue

        lang_dataset = PolarizationDataset(
            df_lang['text'].tolist(),
            df_lang[task_columns].values.tolist(),
            tokenizer
        )

        output = trainer.predict(lang_dataset)

        probs = 1 / (1 + np.exp(-output.predictions))

        polar_preds = (probs[:, 0] > 0.5).astype(int)
        expert_preds_raw = (probs[:, 1:] > threshold).astype(int)

        expert_preds_masked = expert_preds_raw * polar_preds[:, None]
        final_preds = np.column_stack((polar_preds, expert_preds_masked))
        labels = output.label_ids

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            f1_polar = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro', zero_division=0)
            f1_topic = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro', zero_division=0)
            f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro', zero_division=0)
            f1_all = f1_score(labels, final_preds, average='macro', zero_division=0)

            polar_rate = np.mean(polar_preds)

        results.append({
            'language': lang,
            'samples': len(df_lang),
            'f1_polar': f1_polar,
            'f1_topic': f1_topic,
            'f1_toxic': f1_toxic,
            'f1_all': f1_all,
            'gate_open_rate': polar_rate
        })

        print(f"{lang.upper():<5} | {len(df_lang):<5} | {f1_polar:.4f}   | {f1_toxic:.4f}   | {f1_all:.4f}    | {polar_rate:.1%}")

    print("=" * 120)

    return pd.DataFrame(results).sort_values('f1_all', ascending=False)


detailed_results = evaluate_comprehensive(
    trainer,
    val,
    languages,
    task_columns,
    tokenizer,
    threshold=0.5
)

print("\n--- AVERAGES ---")
print(f"Polar:  {detailed_results['f1_polar'].mean():.4f}")
print(f"Topic: {detailed_results['f1_topic'].mean():.4f}")
print(f"Toxic: {detailed_results['f1_toxic'].mean():.4f}")

## Save

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [ ]:
drive_folder = "/content/drive/MyDrive/SemEvalMultiTaskResults/Tabelas/Tabela 1/NG-XLM-RoBERTa-19L-T123-SLv2"

if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)

In [ ]:
csv_path = os.path.join(drive_folder, "detailed_results.csv")
detailed_results.to_csv(csv_path, index=False)

In [ ]:
import time

In [ ]:
print(f"Saving model to {drive_folder}...")
trainer.save_model(drive_folder)
tokenizer.save_pretrained(drive_folder)
print("Model saved.")

print("Waiting 180 seconds for Drive background sync to finish...")
time.sleep(180)

if os.path.exists(os.path.join(drive_folder, "config.json")):
    print("SUCCESS: Config file verified. Safe to shutdown.")
    print("Shutting down runtime now, everything good")
    runtime.unassign()
else:
    print("WARNING: Something went wrong. File not found after waiting.")

print("Shutting down runtime now, no model saved :/")
runtime.unassign()

# Non-Gated XLM-RoBERTa 18L + T123 (Specific Loss)

## Unzip

In [ ]:
!unzip test_phase.zip

## Imports

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.49.0

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score
import numpy as np
import re
import os
import html
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    XLMRobertaModel,
    XLMRobertaPreTrainedModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from torch.utils.data import Dataset
import shutil
from google.colab import files, drive, runtime
import wandb

wandb.init(mode="disabled")

drive.mount('/content/drive')

## Dataset prep

In [ ]:
task_columns = ["polarization", "political", "racial/ethnic", "religious", "gender/sexual", "other", "stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]

task1_columns = ["polarization"]
task2_columns = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
task3_columns = ["stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]


In [ ]:
languages = [
    'arb', 'ben', 'eng', 'hau', 'hin', 'nep', 'urd', 'zho', 'amh',
    'ori', 'fas', 'pol', 'pan', 'rus', 'spa', 'swa', 'tel', 'tur',
    ]
#the first and good result had no 'deu', 'ita', 'khm'
#'hau' is going bad too in the first training section

In [ ]:
def load_and_merge_robust(lang, split_name):
    """
    Loads Subtask 1 (Mandatory).
    If Subtask 2 or 3 are missing, fills their columns with 0.
    """
    # -------------------------------------------------------
    # A. Load Subtask 1 (The Anchor)
    # -------------------------------------------------------
    path_t1 = f'/content/subtask1/{split_name}/{lang}.csv'

    # If Task 1 is missing, we really can't do anything. Skip.
    if not os.path.exists(path_t1):
        if split_name == 'train': # Only warn for train, dev might be partial
            print(f"Skipping {lang.upper()}: No Subtask 1 file found.")
        return None

    # Load the anchor dataframe
    df = pd.read_csv(path_t1)

    # -------------------------------------------------------
    # B. Load Subtask 2 (Topic)
    # -------------------------------------------------------
    path_t2 = f'/content/subtask2/{split_name}/{lang}.csv'

    if os.path.exists(path_t2):
        df_t2 = pd.read_csv(path_t2)
        # LEFT MERGE: Keep all rows from Task 1. Match IDs from Task 2.
        df = df.merge(df_t2, on=['id', 'text'], how='left')
    else:
        # File missing? Fill columns with 0
        print(f"  -> {lang.upper()} missing Subtask 2. Filling columns with 0.")
        for col in task2_columns:
            df[col] = 0

    # -------------------------------------------------------
    # C. Load Subtask 3 (Critical Step for your missing files)
    # -------------------------------------------------------
    path_t3 = f'/content/subtask3/{split_name}/{lang}.csv'

    if os.path.exists(path_t3):
        df_t3 = pd.read_csv(path_t3)
        # LEFT MERGE again
        df = df.merge(df_t3, on=['id', 'text'], how='left')
    else:
        # This will trigger for ITA, MYA, POL, RUS
        print(f"  -> {lang.upper()} missing Subtask 3. Filling columns with 0.")
        for col in task3_columns:
            df[col] = 0

    # -------------------------------------------------------
    # D. Final Cleanup
    # -------------------------------------------------------
    # The merge might create NaNs if IDs didn't match perfectly. Fill them.
    df = df.fillna(0)

    # Tag the language
    df['language'] = lang

    return df

In [ ]:
df_train = None
df_val = None
train_frames = []
val_frames = []
for lang in languages:
    # Load Train
    df_train_lang = load_and_merge_robust(lang, 'train')
    if df_train_lang is not None:
        train_frames.append(df_train_lang)

    # Load Dev
    df_val_lang = load_and_merge_robust(lang, 'dev')
    if df_val_lang is not None:
        val_frames.append(df_val_lang)

In [ ]:
train = pd.concat(train_frames, ignore_index=True)
val = pd.concat(val_frames, ignore_index=True)

In [ ]:
train['text'] = train['text'].astype(str)
val['text'] = val['text'].astype(str)

train = train[train['text'].str.strip().str.len() > 0]
val = val[val['text'].str.strip().str.len() > 0]

print(f"Cleaned Train Shape: {train.shape}")
print(f"Cleaned Val Shape: {val.shape}")

In [ ]:
# Fix the dataset class by inheriting from torch.utils.data.Dataset
class PolarizationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length # Store max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding=False, max_length=self.max_length, return_tensors='pt')

        # Ensure consistent tensor conversion for all items
        item = {key: encoding[key].squeeze() for key in encoding.keys()}
        # CHANGE THIS LINE: Use torch.float instead of torch.long for multi-label classification
        item['labels'] = torch.tensor(label, dtype=torch.float)
        return item

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-large', num_labels=12)

# Create train and Test dataset for multilabel
train_dataset = PolarizationDataset(train['text'].tolist(), train[task_columns].values.tolist(), tokenizer)
val_dataset = PolarizationDataset(val['text'].tolist(), val[task_columns].values.tolist(), tokenizer)

## Model config

In [ ]:
class XLMRobertaForMultiLabelClassification(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)

        # 1. Gate (Polarization)
        self.polar_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1)
        )

        # 2. Topic Head
        self.topic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 5)
        )

        # 3. Toxic Head
        self.toxic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 6)
        )

        self.loss_polar_fct = nn.BCEWithLogitsLoss()
        self.loss_topic_fct = nn.BCEWithLogitsLoss()
        self.loss_toxic_fct = nn.BCEWithLogitsLoss()

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        # Forward Heads
        polar_logit = self.polar_classifier(pooler_output)
        topic_logits = self.topic_classifier(pooler_output)
        toxic_logits = self.toxic_classifier(pooler_output)

        all_logits = torch.cat((polar_logit, topic_logits, toxic_logits), dim=1)

        loss = None
        if labels is not None:
            loss_polar = self.loss_polar_fct(polar_logit, labels[:, 0].unsqueeze(1))
            loss_topic = self.loss_topic_fct(topic_logits, labels[:, 1:6])
            loss_toxic = self.loss_toxic_fct(toxic_logits, labels[:, 6:])

            loss = loss_polar + loss_topic + loss_toxic

        return {"loss": loss, "logits": all_logits}

In [ ]:
# Load the model
model = XLMRobertaForMultiLabelClassification.from_pretrained('xlm-roberta-large', num_labels=12) # use 6 labels

## Training config

In [ ]:
# Define training arguments
training_args = TrainingArguments(
        num_train_epochs=6,              # Reduced from 10 (Critical)
        warmup_ratio=0.1,                # Slightly higher warmup for stability
        learning_rate=2e-5,            # Slightly lower peak LR for "Large" model
        weight_decay=0.01,

        per_device_train_batch_size=32,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=128,

        bf16=True,                       # Native A100 precision
        tf32=True,                       # TensorFloat-32 (Fast Matrix Math)
        optim="adamw_torch_fused",            # Faster compilation

        eval_strategy="steps",
        eval_steps = 200,
        save_strategy="steps",
        save_steps = 200,
        logging_steps=50,

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        dataloader_num_workers=8,
        dataloader_pin_memory=True,
        seed=42
)

def compute_metrics_gated(p):
    logits = p.predictions
    labels = p.label_ids

    probs = 1 / (1 + np.exp(-logits))

    gate_probs = probs[:, 0]
    expert_probs = probs[:, 1:]

    gate_preds = (gate_probs > 0.5).astype(int)
    expert_preds_raw = (expert_probs > 0.5).astype(int)

    expert_preds_masked = expert_preds_raw * gate_preds[:, None]

    final_preds = np.column_stack((gate_preds, expert_preds_masked))

    # Definição dos índices das Tasks
    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]
    idx_t3 = [6, 7, 8, 9, 10, 11]

    # Cálculo dos F1 Scores Específicos
    f1_all = f1_score(labels, final_preds, average='macro')

    f1_polar = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro')

    # Task 2: Topics (Macro)
    f1_topics = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro')

    f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro')

    return {
        'f1_macro': f1_all,
        'f1_polar': f1_polar,
        'f1_topics': f1_topics,
        'f1_toxic': f1_toxic
    }

In [ ]:
def get_llrd_optimizer_parameters(model, learning_rate, weight_decay, lr_decay):
    # 1. Define distinct groups
    # The "Heads" get the Peak Learning Rate
    head_names = ["polar_classifier", "topic_classifier", "toxic_classifier"]

    # Standard exceptions (no weight decay for bias/LayerNorm)
    no_decay = ["bias", "LayerNorm.weight"]

    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
            "lr": learning_rate, # Peak LR (e.g., 2e-5)
        },
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": learning_rate,
        },
    ]

    # 2. Iterate Backwards through the XLM-R Encoder Layers
    # Start from top layer (closer to head) -> go down to embeddings
    layers = [model.roberta.embeddings] + list(model.roberta.encoder.layer)
    layers.reverse() # Reverse so layer 23 is first, embedding is last

    lr = learning_rate
    for layer in layers:
        lr *= lr_decay # Decay the LR (e.g., * 0.95)

        optimizer_grouped_parameters += [
            {
                "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
                "weight_decay": weight_decay,
                "lr": lr,
            },
            {
                "params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
                "lr": lr,
            },
        ]

    return optimizer_grouped_parameters

opt_parameters = get_llrd_optimizer_parameters(model, learning_rate=2e-5, weight_decay=0.01, lr_decay=0.95)
optimizer = torch.optim.AdamW(opt_parameters, lr=2e-5)

## Train

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_gated,  # Use the new metrics function
    data_collator=DataCollatorWithPadding(tokenizer),
    optimizers=(optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)]
)

In [ ]:
# Train the model
trainer.train()

## Evaluation

In [ ]:
results = trainer.evaluate()
print(f"Macro F1 score: {results['eval_f1_macro']}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import warnings

def evaluate_comprehensive(trainer, df_val, languages, task_columns, tokenizer, threshold=0.5):
    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]
    idx_t3 = [6, 7, 8, 9, 10, 11]

    results = []

    print("=" * 120)
    print(f"{'LANG':<5} | {'N':<5} | {'POLAR F1':<8} | {'TOPIC F1':<8} | {'ALL MACRO':<9} | {'GATE RATE':<9}")
    print("=" * 120)

    # 2. Loop per Language
    for lang in languages:
        df_lang = df_val[df_val['language'] == lang]
        if len(df_lang) == 0: continue

        lang_dataset = PolarizationDataset(
            df_lang['text'].tolist(),
            df_lang[task_columns].values.tolist(),
            tokenizer
        )

        output = trainer.predict(lang_dataset)

        probs = 1 / (1 + np.exp(-output.predictions))

        polar_preds = (probs[:, 0] > 0.5).astype(int)
        expert_preds_raw = (probs[:, 1:] > threshold).astype(int)

        expert_preds_masked = expert_preds_raw * polar_preds[:, None]
        final_preds = np.column_stack((polar_preds, expert_preds_masked))
        labels = output.label_ids

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            f1_polar = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro', zero_division=0)
            f1_topic = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro', zero_division=0)
            f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro', zero_division=0)
            f1_all = f1_score(labels, final_preds, average='macro', zero_division=0)

            polar_rate = np.mean(polar_preds)

        results.append({
            'language': lang,
            'samples': len(df_lang),
            'f1_polar': f1_polar,
            'f1_topic': f1_topic,
            'f1_toxic': f1_toxic,
            'f1_all': f1_all,
            'gate_open_rate': polar_rate
        })

        print(f"{lang.upper():<5} | {len(df_lang):<5} | {f1_polar:.4f}   | {f1_toxic:.4f}   | {f1_all:.4f}    | {polar_rate:.1%}")

    print("=" * 120)

    return pd.DataFrame(results).sort_values('f1_all', ascending=False)


detailed_results = evaluate_comprehensive(
    trainer,
    val,
    languages,
    task_columns,
    tokenizer,
    threshold=0.5
)

print("\n--- AVERAGES ---")
print(f"Polar:  {detailed_results['f1_polar'].mean():.4f}")
print(f"Topic: {detailed_results['f1_topic'].mean():.4f}")
print(f"Toxic: {detailed_results['f1_toxic'].mean():.4f}")

## Save

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [ ]:
drive_folder = "/content/drive/MyDrive/SemEvalMultiTaskResults/Tabelas/Tabela 1/NG-XLM-RoBERTa-18L-T123-SL"

if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)

In [ ]:
csv_path = os.path.join(drive_folder, "detailed_results.csv")
detailed_results.to_csv(csv_path, index=False)

In [ ]:
import time

In [ ]:
print(f"Saving model to {drive_folder}...")
trainer.save_model(drive_folder)
tokenizer.save_pretrained(drive_folder)
print("Model saved.")

print("Waiting 180 seconds for Drive background sync to finish...")
time.sleep(180)

if os.path.exists(os.path.join(drive_folder, "config.json")):
    print("SUCCESS: Config file verified. Safe to shutdown.")
    print("Shutting down runtime now, everything good")
    runtime.unassign()
else:
    print("WARNING: Something went wrong. File not found after waiting.")

print("Shutting down runtime now, no model saved :/")
runtime.unassign()

# Non-Gated XLM-RoBERTa 19L + T123 (Specific Loss) + Técnicas (Positional Weighting)

## Unzip

In [ ]:
!unzip test_phase.zip

## Imports

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.49.0

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score
import numpy as np
import re
import os
import html
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    XLMRobertaModel,
    XLMRobertaPreTrainedModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from torch.utils.data import Dataset
import shutil
from google.colab import files, drive, runtime
import wandb

wandb.init(mode="disabled")

drive.mount('/content/drive')

## Dataset prep

In [ ]:
task_columns = ["polarization", "political", "racial/ethnic", "religious", "gender/sexual", "other", "stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]

task1_columns = ["polarization"]
task2_columns = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
task3_columns = ["stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]


In [ ]:
languages = [
    'arb', 'ben', 'mya', 'eng', 'hau', 'hin', 'nep', 'urd', 'zho', 'amh',
    'ori', 'fas', 'pol', 'pan', 'rus', 'spa', 'swa', 'tel', 'tur',
    ]
#the first and good result had no 'deu', 'ita', 'khm'
#'hau' is going bad too in the first training section

In [ ]:
def load_and_merge_robust(lang, split_name):
    """
    Loads Subtask 1 (Mandatory).
    If Subtask 2 or 3 are missing, fills their columns with 0.
    """
    # -------------------------------------------------------
    # A. Load Subtask 1 (The Anchor)
    # -------------------------------------------------------
    path_t1 = f'/content/subtask1/{split_name}/{lang}.csv'

    # If Task 1 is missing, we really can't do anything. Skip.
    if not os.path.exists(path_t1):
        if split_name == 'train': # Only warn for train, dev might be partial
            print(f"Skipping {lang.upper()}: No Subtask 1 file found.")
        return None

    # Load the anchor dataframe
    df = pd.read_csv(path_t1)

    # -------------------------------------------------------
    # B. Load Subtask 2 (Topic)
    # -------------------------------------------------------
    path_t2 = f'/content/subtask2/{split_name}/{lang}.csv'

    if os.path.exists(path_t2):
        df_t2 = pd.read_csv(path_t2)
        # LEFT MERGE: Keep all rows from Task 1. Match IDs from Task 2.
        df = df.merge(df_t2, on=['id', 'text'], how='left')
    else:
        # File missing? Fill columns with 0
        print(f"  -> {lang.upper()} missing Subtask 2. Filling columns with 0.")
        for col in task2_columns:
            df[col] = 0

    # -------------------------------------------------------
    # C. Load Subtask 3 (Critical Step for your missing files)
    # -------------------------------------------------------
    path_t3 = f'/content/subtask3/{split_name}/{lang}.csv'

    if os.path.exists(path_t3):
        df_t3 = pd.read_csv(path_t3)
        # LEFT MERGE again
        df = df.merge(df_t3, on=['id', 'text'], how='left')
    else:
        # This will trigger for ITA, MYA, POL, RUS
        print(f"  -> {lang.upper()} missing Subtask 3. Filling columns with 0.")
        for col in task3_columns:
            df[col] = 0

    # -------------------------------------------------------
    # D. Final Cleanup
    # -------------------------------------------------------
    # The merge might create NaNs if IDs didn't match perfectly. Fill them.
    df = df.fillna(0)

    # Tag the language
    df['language'] = lang

    return df

In [ ]:
df_train = None
df_val = None
train_frames = []
val_frames = []
for lang in languages:
    # Load Train
    df_train_lang = load_and_merge_robust(lang, 'train')
    if df_train_lang is not None:
        train_frames.append(df_train_lang)

    # Load Dev
    df_val_lang = load_and_merge_robust(lang, 'dev')
    if df_val_lang is not None:
        val_frames.append(df_val_lang)

In [ ]:
train = pd.concat(train_frames, ignore_index=True)
val = pd.concat(val_frames, ignore_index=True)

In [ ]:
train['text'] = train['text'].astype(str)
val['text'] = val['text'].astype(str)

train = train[train['text'].str.strip().str.len() > 0]
val = val[val['text'].str.strip().str.len() > 0]

print(f"Cleaned Train Shape: {train.shape}")
print(f"Cleaned Val Shape: {val.shape}")

In [ ]:
expert_cols = task_columns
weights_for_pos = []

print(f"{'LABEL':<20} | {'POSITIVOS (%)':<15} | {'SPARSIDADE (ZEROS)':<20} | {'PESO SUGERIDO'}")
print("-" * 75)
train_non_polarized = train[train['polarization'] == 1]

total_rows = len(train_non_polarized)

for col in expert_cols:
    if col == 'polarization':
        continue
    num_pos = train_non_polarized[col].sum()

    ratio_pos = num_pos / total_rows
    ratio_neg = 1 - ratio_pos

    suggested_weight = ratio_neg / ratio_pos if ratio_pos > 0 else 0

    weights_for_pos.append(suggested_weight)
    print(f"{col:<20} | {ratio_pos*100:.1f}%              | {ratio_neg*100:.1f}%               | {suggested_weight:.1f}")

In [ ]:
# Fix the dataset class by inheriting from torch.utils.data.Dataset
class PolarizationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length # Store max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding=False, max_length=self.max_length, return_tensors='pt')

        # Ensure consistent tensor conversion for all items
        item = {key: encoding[key].squeeze() for key in encoding.keys()}
        # CHANGE THIS LINE: Use torch.float instead of torch.long for multi-label classification
        item['labels'] = torch.tensor(label, dtype=torch.float)
        return item

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-large', num_labels=12)

# Create train and Test dataset for multilabel
train_dataset = PolarizationDataset(train['text'].tolist(), train[task_columns].values.tolist(), tokenizer)
val_dataset = PolarizationDataset(val['text'].tolist(), val[task_columns].values.tolist(), tokenizer)

## Model config

In [ ]:
pos_weight = weights_for_pos

In [ ]:
class XLMRobertaForMultiLabelClassification(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)

        # 1. Gate (Polarization)
        self.polar_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1)
        )

        # 2. Topic Head
        self.topic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 5)
        )

        # 3. Toxic Head
        self.toxic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 6)
        )

        weight_tensor = torch.tensor(pos_weight, dtype=torch.float)

        self.loss_polar_fct = nn.BCEWithLogitsLoss()
        self.loss_topic_fct = nn.BCEWithLogitsLoss(pos_weight=weight_tensor[0:5])
        self.loss_toxic_fct = nn.BCEWithLogitsLoss(pos_weight=weight_tensor[5:])

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        # Forward Heads
        polar_logit = self.polar_classifier(pooler_output)
        topic_logits = self.topic_classifier(pooler_output)
        toxic_logits = self.toxic_classifier(pooler_output)

        all_logits = torch.cat((polar_logit, topic_logits, toxic_logits), dim=1)

        loss = None
        if labels is not None:
            loss_polar = self.loss_polar_fct(polar_logit, labels[:, 0].unsqueeze(1))
            loss_topic = self.loss_topic_fct(topic_logits, labels[:, 1:6])
            loss_toxic = self.loss_toxic_fct(toxic_logits, labels[:, 6:])

            loss = loss_polar + loss_topic + loss_toxic

        return {"loss": loss, "logits": all_logits}

In [ ]:
# Load the model
model = XLMRobertaForMultiLabelClassification.from_pretrained('xlm-roberta-large', num_labels=12) # use 6 labels

## Training config

In [ ]:
# Define training arguments
training_args = TrainingArguments(
        num_train_epochs=6,              # Reduced from 10 (Critical)
        warmup_ratio=0.1,                # Slightly higher warmup for stability
        learning_rate=2e-5,            # Slightly lower peak LR for "Large" model
        weight_decay=0.01,

        per_device_train_batch_size=32,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=128,

        bf16=True,                       # Native A100 precision
        tf32=True,                       # TensorFloat-32 (Fast Matrix Math)
        optim="adamw_torch_fused",            # Faster compilation

        eval_strategy="steps",
        eval_steps = 200,
        save_strategy="steps",
        save_steps = 200,
        logging_steps=50,

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        dataloader_num_workers=8,
        dataloader_pin_memory=True,
        seed=42
)

def compute_metrics_gated(p):
    logits = p.predictions
    labels = p.label_ids

    probs = 1 / (1 + np.exp(-logits))

    gate_probs = probs[:, 0]
    expert_probs = probs[:, 1:]

    gate_preds = (gate_probs > 0.5).astype(int)
    expert_preds_raw = (expert_probs > 0.5).astype(int)

    expert_preds_masked = expert_preds_raw * gate_preds[:, None]

    final_preds = np.column_stack((gate_preds, expert_preds_masked))

    # Definição dos índices das Tasks
    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]
    idx_t3 = [6, 7, 8, 9, 10, 11]

    # Cálculo dos F1 Scores Específicos
    f1_all = f1_score(labels, final_preds, average='macro')

    f1_polar = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro')

    # Task 2: Topics (Macro)
    f1_topics = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro')

    f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro')

    return {
        'f1_macro': f1_all,
        'f1_polar': f1_polar,
        'f1_topics': f1_topics,
        'f1_toxic': f1_toxic
    }

In [ ]:
def get_llrd_optimizer_parameters(model, learning_rate, weight_decay, lr_decay):
    # 1. Define distinct groups
    # The "Heads" get the Peak Learning Rate
    head_names = ["polar_classifier", "topic_classifier", "toxic_classifier"]

    # Standard exceptions (no weight decay for bias/LayerNorm)
    no_decay = ["bias", "LayerNorm.weight"]

    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
            "lr": learning_rate, # Peak LR (e.g., 2e-5)
        },
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": learning_rate,
        },
    ]

    # 2. Iterate Backwards through the XLM-R Encoder Layers
    # Start from top layer (closer to head) -> go down to embeddings
    layers = [model.roberta.embeddings] + list(model.roberta.encoder.layer)
    layers.reverse() # Reverse so layer 23 is first, embedding is last

    lr = learning_rate
    for layer in layers:
        lr *= lr_decay # Decay the LR (e.g., * 0.95)

        optimizer_grouped_parameters += [
            {
                "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
                "weight_decay": weight_decay,
                "lr": lr,
            },
            {
                "params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
                "lr": lr,
            },
        ]

    return optimizer_grouped_parameters

opt_parameters = get_llrd_optimizer_parameters(model, learning_rate=2e-5, weight_decay=0.01, lr_decay=0.95)
optimizer = torch.optim.AdamW(opt_parameters, lr=2e-5)

## Train

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_gated,  # Use the new metrics function
    data_collator=DataCollatorWithPadding(tokenizer),
    optimizers=(optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)]
)

In [ ]:
# Train the model
trainer.train()

## Evaluation

In [ ]:
results = trainer.evaluate()
print(f"Macro F1 score: {results['eval_f1_macro']}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import warnings

def evaluate_comprehensive(trainer, df_val, languages, task_columns, tokenizer, threshold=0.5):
    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]
    idx_t3 = [6, 7, 8, 9, 10, 11]

    results = []

    print("=" * 120)
    print(f"{'LANG':<5} | {'N':<5} | {'POLAR F1':<8} | {'TOPIC F1':<8} | {'ALL MACRO':<9} | {'GATE RATE':<9}")
    print("=" * 120)

    # 2. Loop per Language
    for lang in languages:
        df_lang = df_val[df_val['language'] == lang]
        if len(df_lang) == 0: continue

        lang_dataset = PolarizationDataset(
            df_lang['text'].tolist(),
            df_lang[task_columns].values.tolist(),
            tokenizer
        )

        output = trainer.predict(lang_dataset)

        probs = 1 / (1 + np.exp(-output.predictions))

        polar_preds = (probs[:, 0] > 0.5).astype(int)
        expert_preds_raw = (probs[:, 1:] > threshold).astype(int)

        expert_preds_masked = expert_preds_raw * polar_preds[:, None]
        final_preds = np.column_stack((polar_preds, expert_preds_masked))
        labels = output.label_ids

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            f1_polar = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro', zero_division=0)
            f1_topic = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro', zero_division=0)
            f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro', zero_division=0)
            f1_all = f1_score(labels, final_preds, average='macro', zero_division=0)

            polar_rate = np.mean(polar_preds)

        results.append({
            'language': lang,
            'samples': len(df_lang),
            'f1_polar': f1_polar,
            'f1_topic': f1_topic,
            'f1_toxic': f1_toxic,
            'f1_all': f1_all,
            'gate_open_rate': polar_rate
        })

        print(f"{lang.upper():<5} | {len(df_lang):<5} | {f1_polar:.4f}   | {f1_toxic:.4f}   | {f1_all:.4f}    | {polar_rate:.1%}")

    print("=" * 120)

    return pd.DataFrame(results).sort_values('f1_all', ascending=False)


detailed_results = evaluate_comprehensive(
    trainer,
    val,
    languages,
    task_columns,
    tokenizer,
    threshold=0.5
)

print("\n--- AVERAGES ---")
print(f"Polar:  {detailed_results['f1_polar'].mean():.4f}")
print(f"Topic: {detailed_results['f1_topic'].mean():.4f}")
print(f"Toxic: {detailed_results['f1_toxic'].mean():.4f}")

## Save

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [ ]:
drive_folder = "/content/drive/MyDrive/SemEvalMultiTaskResults/Tabelas/Tabela 1/NG-XLM-RoBERTa-19L-T123-SL-PW"

if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)

In [ ]:
csv_path = os.path.join(drive_folder, "detailed_results.csv")
detailed_results.to_csv(csv_path, index=False)

In [ ]:
import time

In [ ]:
print(f"Saving model to {drive_folder}...")
trainer.save_model(drive_folder)
tokenizer.save_pretrained(drive_folder)
print("Model saved.")

print("Waiting 180 seconds for Drive background sync to finish...")
time.sleep(180)

if os.path.exists(os.path.join(drive_folder, "config.json")):
    print("SUCCESS: Config file verified. Safe to shutdown.")
    print("Shutting down runtime now, everything good")
    runtime.unassign()
else:
    print("WARNING: Something went wrong. File not found after waiting.")

print("Shutting down runtime now, no model saved :/")
runtime.unassign()

# Non-Gated XLM-RoBERTa 19L + T12 (Specific Loss) + Técnicas (Positional Weighting)

## Unzip

In [ ]:
!unzip test_phase.zip

## Imports

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.49.0

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score
import numpy as np
import re
import os
import html
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    XLMRobertaModel,
    XLMRobertaPreTrainedModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    TrainerCallback
)
from torch.utils.data import Dataset
import shutil
from google.colab import files, drive, runtime
import wandb

wandb.init(mode="disabled")

drive.mount('/content/drive')

## Dataset prep

In [ ]:
task_columns = ["polarization", "political", "racial/ethnic", "religious", "gender/sexual", "other"]
task1_columns = ["polarization"]
task2_columns = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
task3_columns = ["stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]


In [ ]:
languages = [
    'arb', 'ben', 'mya', 'eng', 'hin', 'nep', 'urd', 'zho', 'amh', 'hau',
    'ori', 'fas', 'pol', 'pan', 'rus', 'spa', 'swa', 'tel', 'tur'
    ]

In [ ]:
def load_and_merge_robust(lang, split_name):
    """
    Loads Subtask 1 (Mandatory).
    If Subtask 2 or 3 are missing, fills their columns with 0.
    """
    # -------------------------------------------------------
    # A. Load Subtask 1 (The Anchor)
    # -------------------------------------------------------
    path_t1 = f'/content/subtask1/{split_name}/{lang}.csv'

    # If Task 1 is missing, we really can't do anything. Skip.
    if not os.path.exists(path_t1):
        if split_name == 'train': # Only warn for train, dev might be partial
            print(f"Skipping {lang.upper()}: No Subtask 1 file found.")
        return None

    # Load the anchor dataframe
    df = pd.read_csv(path_t1)

    # -------------------------------------------------------
    # B. Load Subtask 2 (Topic)
    # -------------------------------------------------------
    path_t2 = f'/content/subtask2/{split_name}/{lang}.csv'

    if os.path.exists(path_t2):
        df_t2 = pd.read_csv(path_t2)
        # LEFT MERGE: Keep all rows from Task 1. Match IDs from Task 2.
        df = df.merge(df_t2, on=['id', 'text'], how='left')
    else:
        # File missing? Fill columns with 0
        print(f"  -> {lang.upper()} missing Subtask 2. Filling columns with 0.")
        for col in task2_columns:
            df[col] = 0

    # Tag the language
    df['language'] = lang

    return df

In [ ]:
df_train = None
df_val = None
train_frames = []
val_frames = []
for lang in languages:
    # Load Train
    df_train_lang = load_and_merge_robust(lang, 'train')
    if df_train_lang is not None:
        train_frames.append(df_train_lang)

    # Load Dev
    df_val_lang = load_and_merge_robust(lang, 'dev')
    if df_val_lang is not None:
        val_frames.append(df_val_lang)

In [ ]:
train = pd.concat(train_frames, ignore_index=True)
val = pd.concat(val_frames, ignore_index=True)

In [ ]:
train['text'] = train['text'].astype(str)
val['text'] = val['text'].astype(str)

train = train[train['text'].str.strip().str.len() > 0]
val = val[val['text'].str.strip().str.len() > 0]

print(f"Cleaned Train Shape: {train.shape}")
print(f"Cleaned Val Shape: {val.shape}")

In [ ]:
# Defina as colunas da Task 2 (Experts)
expert_cols = task_columns
weights_for_pos = []

print(f"{'LABEL':<20} | {'POSITIVOS (%)':<15} | {'SPARSIDADE (ZEROS)':<20} | {'PESO SUGERIDO'}")
print("-" * 75)

total_rows = len(train) # Assumindo que 'train' é seu dataframe

for col in expert_cols:
    num_pos = train[col].sum()
    ratio_pos = num_pos / total_rows
    ratio_neg = 1 - ratio_pos

    # O peso ideal é geralmente: Negativos / Positivos
    # Isso equilibra a balança para que 1 positivo valha tanto quanto N negativos
    suggested_weight = ratio_neg / ratio_pos if ratio_pos > 0 else 0
    weights_for_pos.append(suggested_weight)
    print(f"{col:<20} | {ratio_pos*100:.1f}%          | {ratio_neg*100:.1f}%              | {suggested_weight:.1f}")
weights_for_pos = weights_for_pos[1:]
print(weights_for_pos)

In [ ]:
for col in task_columns:
    train[col] = train[col].fillna(0).astype(int)
    val[col] = val[col].fillna(0).astype(int)

# Verifica se só tem 0 e 1
for col in task_columns:
    unique_vals = train[col].unique()
    if not all(v in [0, 1] for v in unique_vals):
        print(f"❌ ERRO: {col} tem valores inválidos: {unique_vals}")

In [ ]:
# Fix the dataset class by inheriting from torch.utils.data.Dataset
class PolarizationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length # Store max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding=False, max_length=self.max_length, return_tensors='pt')

        # Ensure consistent tensor conversion for all items
        item = {key: encoding[key].squeeze() for key in encoding.keys()}
        # CHANGE THIS LINE: Use torch.float instead of torch.long for multi-label classification
        item['labels'] = torch.tensor(label, dtype=torch.float)
        return item

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-large', num_labels=6)

# Create train and Test dataset for multilabel
train_dataset = PolarizationDataset(train['text'].tolist(), train[task_columns].values.tolist(), tokenizer)
val_dataset = PolarizationDataset(val['text'].tolist(), val[task_columns].values.tolist(), tokenizer)

## Model config

In [ ]:
pos_weight = weights_for_pos
class XLMRobertaForMultiLabelClassification(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)

        # Gate (Polarization)
        self.polar_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1)
        )

        # Experts (Topics)
        self.topic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 5)
        )

        # Initialize the Loss functions here
        self.loss_polar_fct = nn.BCEWithLogitsLoss()

        # USE FOCAL LOSS FOR EXPERTS ONLY (The hard part)
        self.pos_weight = torch.tensor(pos_weight)
        self.loss_topic_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        # 1. Gate Forward
        polar_logit = self.polar_classifier(pooler_output)
        topic_logits = self.topic_classifier(pooler_output)

        # 3. Loss Calculation
        loss = None
        if labels is not None:
            # Separate Labels
            polar_labels = labels[:, 0].unsqueeze(1)
            topic_labels = labels[:, 1:]

            # A. Calculate Gate Loss (Standard BCE is fine here)
            loss_polar = self.loss_polar_fct(polar_logit, polar_labels)

            # B. Calculate Expert Loss (Using Focal Loss to fix sparsity)
            loss_topic = self.loss_topic_fct(topic_logits, topic_labels)

            # Combine them
            loss = loss_polar + loss_topic

        logits = torch.cat((polar_logit, topic_logits), dim=1)

        return {"loss": loss, "logits": logits}

In [ ]:
# Load the model
model = XLMRobertaForMultiLabelClassification.from_pretrained('xlm-roberta-large', num_labels=6) # use 6 labels

## Training config

In [ ]:
# Define training arguments
training_args = TrainingArguments(
        num_train_epochs=6,              # Reduced from 10 (Critical)
        warmup_ratio=0.1,                # Slightly higher warmup for stability
        learning_rate=2e-5,            # Slightly lower peak LR for "Large" model
        weight_decay=0.01,               # 0.1 is aggressive; 0.05 is standard for Large

        # 2. A100 40GB VRAM Optimization
        per_device_train_batch_size=32,  # 64 is risky for VRAM. 32 is safe.
        gradient_accumulation_steps=2,   # 32 * 2 = 64 Effective Batch Size
        per_device_eval_batch_size=128,   # 32 is safe.

        # 3. Speed & Precision
        bf16=True,                       # Native A100 precision
        tf32=True,                       # TensorFloat-32 (Fast Matrix Math)
        optim="adamw_torch_fused",            # Faster compilation

        # 4. Evaluation Strategy (Catch the peak)
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        logging_steps=50,

        # 5. Standard
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        dataloader_num_workers=8,
        dataloader_pin_memory=True,
        seed=42
)

def compute_metrics_gated(p):
    logits = p.predictions
    labels = p.label_ids

    probs = 1 / (1 + np.exp(-logits))

    gate_probs = probs[:, 0]
    expert_probs = probs[:, 1:]

    gate_preds = (gate_probs > 0.5).astype(int)
    expert_preds_raw = (expert_probs > 0.5).astype(int)

    expert_preds_masked = expert_preds_raw * gate_preds[:, None]

    final_preds = np.column_stack((gate_preds, expert_preds_masked))

    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]

    f1_all = f1_score(labels, final_preds, average='macro')

    f1_gate = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro')

    f1_topics = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro')

    return {
        'f1_macro': f1_all,
        'f1_gate': f1_gate,
        'f1_topics': f1_topics,
    }

In [ ]:
def get_llrd_optimizer_parameters(model, learning_rate, weight_decay, lr_decay):
    # 1. Define distinct groups
    # The "Heads" get the Peak Learning Rate
    head_names = ["polar_classifier", "topic_classifier"]

    # Standard exceptions (no weight decay for bias/LayerNorm)
    no_decay = ["bias", "LayerNorm.weight"]

    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
            "lr": learning_rate, # Peak LR (e.g., 2e-5)
        },
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": learning_rate,
        },
    ]

    # 2. Iterate Backwards through the XLM-R Encoder Layers
    # Start from top layer (closer to head) -> go down to embeddings
    layers = [model.roberta.embeddings] + list(model.roberta.encoder.layer)
    layers.reverse() # Reverse so layer 23 is first, embedding is last

    lr = learning_rate
    for layer in layers:
        lr *= lr_decay # Decay the LR (e.g., * 0.95)

        optimizer_grouped_parameters += [
            {
                "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
                "weight_decay": weight_decay,
                "lr": lr,
            },
            {
                "params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
                "lr": lr,
            },
        ]

    return optimizer_grouped_parameters

opt_parameters = get_llrd_optimizer_parameters(model, learning_rate=2e-5, weight_decay=0.01, lr_decay=0.95)
optimizer = torch.optim.AdamW(opt_parameters, lr=2e-5)

## Train

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_gated,
    data_collator=DataCollatorWithPadding(tokenizer),
    optimizers=(optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4, early_stopping_threshold=0.0001)] #tive que mudar pois o modelo sofre um pouco para convergir no começo
)

In [ ]:
# Train the model
trainer.train()

## Evaluation

In [ ]:
# Evaluate the model on the validation set
results = trainer.evaluate()
print(f"Macro F1 score on validation set for Subtask 3: {results['eval_f1_macro']}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import warnings

def evaluate_comprehensive(trainer, df_val, languages, task_columns, tokenizer, threshold=0.5):
    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]

    results = []

    print("=" * 120)
    print(f"{'LANG':<5} | {'N':<5} | {'POLAR F1':<8} | {'TOPIC F1':<8} | {'ALL MACRO':<9} | {'GATE RATE':<9}")
    print("=" * 120)

    # 2. Loop per Language
    for lang in languages:
        df_lang = df_val[df_val['language'] == lang]
        if len(df_lang) == 0: continue

        lang_dataset = PolarizationDataset(
            df_lang['text'].tolist(),
            df_lang[task_columns].values.tolist(),
            tokenizer
        )

        output = trainer.predict(lang_dataset)

        probs = 1 / (1 + np.exp(-output.predictions))

        polar_preds = (probs[:, 0] > 0.5).astype(int)
        expert_preds_raw = (probs[:, 1:] > threshold).astype(int)

        expert_preds_masked = expert_preds_raw * polar_preds[:, None]
        final_preds = np.column_stack((polar_preds, expert_preds_masked))
        labels = output.label_ids

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            f1_polar = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro', zero_division=0)
            f1_topic = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro', zero_division=0)
            f1_all = f1_score(labels, final_preds, average='macro', zero_division=0)

            polar_rate = np.mean(polar_preds)

        results.append({
            'language': lang,
            'samples': len(df_lang),
            'f1_polar': f1_polar,
            'f1_topic': f1_topic,
            'f1_all': f1_all,
            'gate_open_rate': polar_rate
        })

        print(f"{lang.upper():<5} | {len(df_lang):<5} | {f1_polar:.4f}   | {f1_topic:.4f}   | {f1_all:.4f}    | {polar_rate:.1%}")

    print("=" * 120)

    return pd.DataFrame(results).sort_values('f1_all', ascending=False)


detailed_results = evaluate_comprehensive(
    trainer,
    val,
    languages,
    task_columns,
    tokenizer,
    threshold=0.5
)

print("\n--- AVERAGES ---")
print(f"Polar:  {detailed_results['f1_polar'].mean():.4f}")
print(f"Topic: {detailed_results['f1_topic'].mean():.4f}")

## Save

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [ ]:
drive_folder = "/content/drive/MyDrive/SemEvalMultiTaskResults/Tabelas/Tabela 2/NG-XLM-RoBERTa-19L-T12-SL-PW"

if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)

In [ ]:
csv_path = os.path.join(drive_folder, "detailed_results.csv")
detailed_results.to_csv(csv_path, index=False)

In [ ]:
import time

In [ ]:
print(f"Saving model to {drive_folder}...")
trainer.save_model(drive_folder)
tokenizer.save_pretrained(drive_folder)
print("Model saved.")

print("Waiting 180 seconds for Drive background sync to finish...")
time.sleep(180)

if os.path.exists(os.path.join(drive_folder, "config.json")):
    print("SUCCESS: Config file verified. Safe to shutdown.")
    print("Shutting down runtime now, everything good")
    runtime.unassign()
else:
    print("WARNING: Something went wrong. File not found after waiting.")

print("Shutting down runtime now, no model saved :/")
runtime.unassign()

# Non-Gated XLM-RoBERTa 19L + T13 (Specific Loss) + Técnicas (Positional Weighting)

## Unzip

In [ ]:
!unzip test_phase.zip

## Imports

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.49.0

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score
import numpy as np
import re
import os
import html
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    XLMRobertaModel,
    XLMRobertaPreTrainedModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    TrainerCallback
)
from torch.utils.data import Dataset
import shutil
from google.colab import files, drive, runtime
import wandb

wandb.init(mode="disabled")

drive.mount('/content/drive')

## Dataset prep

In [ ]:
task_columns = ["polarization", "stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]
task1_columns = ["polarization"]
task2_columns = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
task3_columns = ["stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]


In [ ]:
languages = [
    'arb', 'ben', 'mya', 'eng', 'hau', 'hin', 'nep', 'urd', 'zho', 'amh',
    'ori', 'fas', 'pol', 'pan', 'rus', 'spa', 'swa', 'tel', 'tur',
    ]

In [ ]:
def load_and_merge_robust(lang, split_name):
    """
    Loads Subtask 1 (Mandatory).
    If Subtask 2 or 3 are missing, fills their columns with 0.
    """
    # -------------------------------------------------------
    # A. Load Subtask 1 (The Anchor)
    # -------------------------------------------------------
    path_t1 = f'/content/subtask1/{split_name}/{lang}.csv'

    # If Task 1 is missing, we really can't do anything. Skip.
    if not os.path.exists(path_t1):
        if split_name == 'train': # Only warn for train, dev might be partial
            print(f"Skipping {lang.upper()}: No Subtask 1 file found.")
        return None

    # Load the anchor dataframe
    df = pd.read_csv(path_t1)

    # -------------------------------------------------------
    # B. Load Subtask 2 (Topic)
    # -------------------------------------------------------
    path_t2 = f'/content/subtask3/{split_name}/{lang}.csv'

    if os.path.exists(path_t2):
        df_t3 = pd.read_csv(path_t2)
        # LEFT MERGE: Keep all rows from Task 1. Match IDs from Task 2.
        df = df.merge(df_t3, on=['id', 'text'], how='left')
    else:
        # File missing? Fill columns with 0
        print(f"  -> {lang.upper()} missing Subtask 3. Filling columns with 0.")
        for col in task3_columns:
            df[col] = 0

    # Tag the language
    df['language'] = lang

    return df

In [ ]:
df_train = None
df_val = None
train_frames = []
val_frames = []
for lang in languages:
    # Load Train
    df_train_lang = load_and_merge_robust(lang, 'train')
    if df_train_lang is not None:
        train_frames.append(df_train_lang)

    # Load Dev
    df_val_lang = load_and_merge_robust(lang, 'dev')
    if df_val_lang is not None:
        val_frames.append(df_val_lang)

In [ ]:
train = pd.concat(train_frames, ignore_index=True)
val = pd.concat(val_frames, ignore_index=True)

In [ ]:
train['text'] = train['text'].astype(str)
val['text'] = val['text'].astype(str)

train = train[train['text'].str.strip().str.len() > 0]
val = val[val['text'].str.strip().str.len() > 0]

print(f"Cleaned Train Shape: {train.shape}")
print(f"Cleaned Val Shape: {val.shape}")

In [ ]:
# Defina as colunas da Task 2 (Experts)
expert_cols = task_columns
weights_for_pos = []

print(f"{'LABEL':<20} | {'POSITIVOS (%)':<15} | {'SPARSIDADE (ZEROS)':<20} | {'PESO SUGERIDO'}")
print("-" * 75)

total_rows = len(train) # Assumindo que 'train' é seu dataframe

for col in expert_cols:
    num_pos = train[col].sum()
    ratio_pos = num_pos / total_rows
    ratio_neg = 1 - ratio_pos

    # O peso ideal é geralmente: Negativos / Positivos
    # Isso equilibra a balança para que 1 positivo valha tanto quanto N negativos
    suggested_weight = ratio_neg / ratio_pos if ratio_pos > 0 else 0
    weights_for_pos.append(suggested_weight)
    print(f"{col:<20} | {ratio_pos*100:.1f}%          | {ratio_neg*100:.1f}%              | {suggested_weight:.1f}")
weights_for_pos = weights_for_pos[1:]
print(weights_for_pos)

In [ ]:
for col in task_columns:
    train[col] = train[col].fillna(0).astype(int)
    val[col] = val[col].fillna(0).astype(int)

# Verifica se só tem 0 e 1
for col in task_columns:
    unique_vals = train[col].unique()
    if not all(v in [0, 1] for v in unique_vals):
        print(f"❌ ERRO: {col} tem valores inválidos: {unique_vals}")

In [ ]:
# Fix the dataset class by inheriting from torch.utils.data.Dataset
class PolarizationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length # Store max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding=False, max_length=self.max_length, return_tensors='pt')

        # Ensure consistent tensor conversion for all items
        item = {key: encoding[key].squeeze() for key in encoding.keys()}
        # CHANGE THIS LINE: Use torch.float instead of torch.long for multi-label classification
        item['labels'] = torch.tensor(label, dtype=torch.float)
        return item

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-large', num_labels=7)

train_dataset = PolarizationDataset(train['text'].tolist(), train[task_columns].values.tolist(), tokenizer)
val_dataset = PolarizationDataset(val['text'].tolist(), val[task_columns].values.tolist(), tokenizer)

## Model config

In [ ]:
pos_weight = weights_for_pos
class XLMRobertaForMultiLabelClassification(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)

        # Gate (Polarization)
        self.polar_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1)
        )

        # Experts (Topics)
        self.toxic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 6)
        )

        # Initialize the Loss functions here
        self.loss_polar_fct = nn.BCEWithLogitsLoss()

        # USE FOCAL LOSS FOR EXPERTS ONLY (The hard part)
        self.pos_weight = torch.tensor(pos_weight)
        self.loss_toxic_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        # 1. Gate Forward
        polar_logit = self.polar_classifier(pooler_output)
        toxic_logits = self.toxic_classifier(pooler_output)

        # 3. Loss Calculation
        loss = None
        if labels is not None:
            # Separate Labels
            polar_labels = labels[:, 0].unsqueeze(1)
            toxic_labels = labels[:, 1:]

            # A. Calculate Gate Loss (Standard BCE is fine here)
            loss_polar = self.loss_polar_fct(polar_logit, polar_labels)

            # B. Calculate Expert Loss (Using Focal Loss to fix sparsity)
            loss_toxic = self.loss_toxic_fct(toxic_logits, toxic_labels)

            # Combine them
            loss = loss_polar + loss_toxic

        logits = torch.cat((polar_logit, toxic_logits), dim=1)

        return {"loss": loss, "logits": logits}

In [ ]:
# Load the model
model = XLMRobertaForMultiLabelClassification.from_pretrained('xlm-roberta-large', num_labels=7) # use 6 labels

## Training config

In [ ]:
# Define training arguments
training_args = TrainingArguments(
        num_train_epochs=6,              # Reduced from 10 (Critical)
        warmup_ratio=0.1,                # Slightly higher warmup for stability
        learning_rate=2e-5,            # Slightly lower peak LR for "Large" model
        weight_decay=0.01,               # 0.1 is aggressive; 0.05 is standard for Large

        # 2. A100 40GB VRAM Optimization
        per_device_train_batch_size=32,  # 64 is risky for VRAM. 32 is safe.
        gradient_accumulation_steps=2,   # 32 * 2 = 64 Effective Batch Size
        per_device_eval_batch_size=128,   # 32 is safe.

        # 3. Speed & Precision
        bf16=True,                       # Native A100 precision
        tf32=True,                       # TensorFloat-32 (Fast Matrix Math)
        optim="adamw_torch_fused",            # Faster compilation

        # 4. Evaluation Strategy (Catch the peak)
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        logging_steps=50,

        # 5. Standard
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        dataloader_num_workers=8,
        dataloader_pin_memory=True,
        seed=42
)

def compute_metrics_gated(p):
    logits = p.predictions
    labels = p.label_ids

    probs = 1 / (1 + np.exp(-logits))

    gate_probs = probs[:, 0]
    expert_probs = probs[:, 1:]

    gate_preds = (gate_probs > 0.5).astype(int)
    expert_preds_raw = (expert_probs > 0.5).astype(int)

    expert_preds_masked = expert_preds_raw * gate_preds[:, None]

    final_preds = np.column_stack((gate_preds, expert_preds_masked))

    idx_t1 = [0]
    idx_t3 = [1, 2, 3, 4, 5, 6]

    f1_all = f1_score(labels, final_preds, average='macro')

    f1_gate = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro')

    f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro')

    return {
        'f1_macro': f1_all,
        'f1_gate': f1_gate,
        'f1_toxic': f1_toxic,
    }

In [ ]:
def get_llrd_optimizer_parameters(model, learning_rate, weight_decay, lr_decay):
    head_names = ["polar_classifier", "toxic_classifier"]

    # Standard exceptions (no weight decay for bias/LayerNorm)
    no_decay = ["bias", "LayerNorm.weight"]

    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
            "lr": learning_rate, # Peak LR (e.g., 2e-5)
        },
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": learning_rate,
        },
    ]

    # 2. Iterate Backwards through the XLM-R Encoder Layers
    # Start from top layer (closer to head) -> go down to embeddings
    layers = [model.roberta.embeddings] + list(model.roberta.encoder.layer)
    layers.reverse() # Reverse so layer 23 is first, embedding is last

    lr = learning_rate
    for layer in layers:
        lr *= lr_decay # Decay the LR (e.g., * 0.95)

        optimizer_grouped_parameters += [
            {
                "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
                "weight_decay": weight_decay,
                "lr": lr,
            },
            {
                "params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
                "lr": lr,
            },
        ]

    return optimizer_grouped_parameters

opt_parameters = get_llrd_optimizer_parameters(model, learning_rate=2e-5, weight_decay=0.01, lr_decay=0.95)
optimizer = torch.optim.AdamW(opt_parameters, lr=2e-5)

## Train

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_gated,
    data_collator=DataCollatorWithPadding(tokenizer),
    optimizers=(optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)]
)

In [ ]:
# Train the model
trainer.train()

## Evaluation

In [ ]:
# Evaluate the model on the validation set
results = trainer.evaluate()
print(f"Macro F1 score: {results['eval_f1_macro']}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import warnings

def evaluate_comprehensive(trainer, df_val, languages, task_columns, tokenizer, threshold=0.5):
    idx_t1 = [0]
    idx_t3 = [1, 2, 3, 4, 5, 6]

    results = []

    print("=" * 120)
    print(f"{'LANG':<5} | {'N':<5} | {'POLAR F1':<8} | {'TOXIC F1':<8} | {'ALL MACRO':<9} | {'GATE RATE':<9}")
    print("=" * 120)

    # 2. Loop per Language
    for lang in languages:
        df_lang = df_val[df_val['language'] == lang]
        if len(df_lang) == 0: continue

        lang_dataset = PolarizationDataset(
            df_lang['text'].tolist(),
            df_lang[task_columns].values.tolist(),
            tokenizer
        )

        output = trainer.predict(lang_dataset)

        probs = 1 / (1 + np.exp(-output.predictions))

        polar_preds = (probs[:, 0] > 0.5).astype(int)
        expert_preds_raw = (probs[:, 1:] > threshold).astype(int)

        expert_preds_masked = expert_preds_raw * polar_preds[:, None]
        final_preds = np.column_stack((polar_preds, expert_preds_masked))
        labels = output.label_ids

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            f1_polar = f1_score(labels[:, idx_t1], final_preds[:, idx_t1], average='macro', zero_division=0)
            f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro', zero_division=0)
            f1_all = f1_score(labels, final_preds, average='macro', zero_division=0)

            polar_rate = np.mean(polar_preds)

        results.append({
            'language': lang,
            'samples': len(df_lang),
            'f1_polar': f1_polar,
            'f1_toxic': f1_toxic,
            'f1_all': f1_all,
            'gate_open_rate': polar_rate
        })

        print(f"{lang.upper():<5} | {len(df_lang):<5} | {f1_polar:.4f}   | {f1_toxic:.4f}   | {f1_all:.4f}    | {polar_rate:.1%}")

    print("=" * 120)

    return pd.DataFrame(results).sort_values('f1_all', ascending=False)


detailed_results = evaluate_comprehensive(
    trainer,
    val,
    languages,
    task_columns,
    tokenizer,
    threshold=0.5
)

print("\n--- AVERAGES ---")
print(f"Polar:  {detailed_results['f1_polar'].mean():.4f}")
print(f"Toxic: {detailed_results['f1_toxic'].mean():.4f}")

## Save

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [ ]:
drive_folder = "/content/drive/MyDrive/SemEvalMultiTaskResults/Tabelas/Tabela 2/NG-XLM-RoBERTa-19L-T13-SL-PW"

if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)

In [ ]:
csv_path = os.path.join(drive_folder, "detailed_results.csv")
detailed_results.to_csv(csv_path, index=False)

In [ ]:
import time

In [ ]:
print(f"Saving model to {drive_folder}...")
trainer.save_model(drive_folder)
tokenizer.save_pretrained(drive_folder)
print("Model saved.")

print("Waiting 180 seconds for Drive background sync to finish...")
time.sleep(180)

if os.path.exists(os.path.join(drive_folder, "config.json")):
    print("SUCCESS: Config file verified. Safe to shutdown.")
    print("Shutting down runtime now, everything good")
    runtime.unassign()
else:
    print("WARNING: Something went wrong. File not found after waiting.")

print("Shutting down runtime now, no model saved :/")
runtime.unassign()

# Non-Gated XLM-RoBERTa 19L + T23 (Specific Loss) + Técnicas (Positional Weighting)

## Unzip

In [ ]:
!unzip test_phase.zip

## Imports

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.49.0

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score
import numpy as np
import re
import os
import html
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    XLMRobertaModel,
    XLMRobertaPreTrainedModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from torch.utils.data import Dataset
import shutil
from google.colab import files, drive, runtime
import wandb

wandb.init(mode="disabled")

drive.mount('/content/drive')

## Dataset prep

In [ ]:
task_columns = ["polarization", "political", "racial/ethnic", "religious", "gender/sexual", "other", "stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]
expert_columns = ["political", "racial/ethnic", "religious", "gender/sexual", "other", "stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]
task1_columns = ["polarization"]
task2_columns = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
task3_columns = ["stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]


In [ ]:
languages = [
    'arb', 'ben', 'mya', 'eng', 'hau', 'hin', 'nep', 'urd', 'zho', 'amh',
    'ori', 'fas', 'pol', 'pan', 'rus', 'spa', 'swa', 'tel', 'tur',
    ]
#the first and good result had no 'deu', 'ita', 'khm'
#'hau' is going bad too in the first training section

In [ ]:
def load_and_merge_robust(lang, split_name):
    """
    Loads Subtask 1 (Mandatory).
    If Subtask 2 or 3 are missing, fills their columns with 0.
    """
    # -------------------------------------------------------
    # A. Load Subtask 1 (The Anchor)
    # -------------------------------------------------------
    path_t1 = f'/content/subtask1/{split_name}/{lang}.csv'

    # If Task 1 is missing, we really can't do anything. Skip.
    if not os.path.exists(path_t1):
        if split_name == 'train': # Only warn for train, dev might be partial
            print(f"Skipping {lang.upper()}: No Subtask 1 file found.")
        return None

    # Load the anchor dataframe
    df = pd.read_csv(path_t1)

    # -------------------------------------------------------
    # B. Load Subtask 2 (Topic)
    # -------------------------------------------------------
    path_t2 = f'/content/subtask2/{split_name}/{lang}.csv'

    if os.path.exists(path_t2):
        df_t2 = pd.read_csv(path_t2)
        # LEFT MERGE: Keep all rows from Task 1. Match IDs from Task 2.
        df = df.merge(df_t2, on=['id', 'text'], how='left')
    else:
        # File missing? Fill columns with 0
        print(f"  -> {lang.upper()} missing Subtask 2. Filling columns with 0.")
        for col in task2_columns:
            df[col] = 0

    # -------------------------------------------------------
    # C. Load Subtask 3 (Critical Step for your missing files)
    # -------------------------------------------------------
    path_t3 = f'/content/subtask3/{split_name}/{lang}.csv'

    if os.path.exists(path_t3):
        df_t3 = pd.read_csv(path_t3)
        # LEFT MERGE again
        df = df.merge(df_t3, on=['id', 'text'], how='left')
    else:
        # This will trigger for ITA, MYA, POL, RUS
        print(f"  -> {lang.upper()} missing Subtask 3. Filling columns with 0.")
        for col in task3_columns:
            df[col] = 0

    # -------------------------------------------------------
    # D. Final Cleanup
    # -------------------------------------------------------
    # The merge might create NaNs if IDs didn't match perfectly. Fill them.
    df = df.fillna(0)

    # Tag the language
    df['language'] = lang

    return df

In [ ]:
df_train = None
df_val = None
train_frames = []
val_frames = []
for lang in languages:
    # Load Train
    df_train_lang = load_and_merge_robust(lang, 'train')
    if df_train_lang is not None:
        train_frames.append(df_train_lang)

    # Load Dev
    df_val_lang = load_and_merge_robust(lang, 'dev')
    if df_val_lang is not None:
        val_frames.append(df_val_lang)

In [ ]:
train = pd.concat(train_frames, ignore_index=True)
val = pd.concat(val_frames, ignore_index=True)

In [ ]:
train['text'] = train['text'].astype(str)
val['text'] = val['text'].astype(str)

train = train[train['text'].str.strip().str.len() > 0]
val = val[val['text'].str.strip().str.len() > 0]

print(f"Cleaned Train Shape: {train.shape}")
print(f"Cleaned Val Shape: {val.shape}")

In [ ]:
expert_cols = task_columns
weights_for_pos = []

print(f"{'LABEL':<20} | {'POSITIVOS (%)':<15} | {'SPARSIDADE (ZEROS)':<20} | {'PESO SUGERIDO'}")
print("-" * 75)
train_non_polarized = train[train['polarization'] == 1]

total_rows = len(train_non_polarized)

for col in expert_cols:
    if col == 'polarization':
        continue
    num_pos = train_non_polarized[col].sum()

    ratio_pos = num_pos / total_rows
    ratio_neg = 1 - ratio_pos

    suggested_weight = ratio_neg / ratio_pos if ratio_pos > 0 else 0

    weights_for_pos.append(suggested_weight)
    print(f"{col:<20} | {ratio_pos*100:.1f}%              | {ratio_neg*100:.1f}%               | {suggested_weight:.1f}")

In [ ]:
# Fix the dataset class by inheriting from torch.utils.data.Dataset
class PolarizationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length # Store max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding=False, max_length=self.max_length, return_tensors='pt')

        # Ensure consistent tensor conversion for all items
        item = {key: encoding[key].squeeze() for key in encoding.keys()}
        # CHANGE THIS LINE: Use torch.float instead of torch.long for multi-label classification
        item['labels'] = torch.tensor(label, dtype=torch.float)
        return item

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-large', num_labels=11)

# Create train and Test dataset for multilabel
train_dataset = PolarizationDataset(train['text'].tolist(), train[expert_columns].values.tolist(), tokenizer)
val_dataset = PolarizationDataset(val['text'].tolist(), val[expert_columns].values.tolist(), tokenizer)

## Model config

In [ ]:
pos_weight = weights_for_pos

In [ ]:
class XLMRobertaForMultiLabelClassification(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)

        # 2. Topic Head
        self.topic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 5)
        )

        # 3. Toxic Head
        self.toxic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 6)
        )

        weight_tensor = torch.tensor(pos_weight, dtype=torch.float)

        self.loss_topic_fct = nn.BCEWithLogitsLoss(pos_weight=weight_tensor[0:5])
        self.loss_toxic_fct = nn.BCEWithLogitsLoss(pos_weight=weight_tensor[5:])

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        topic_logits = self.topic_classifier(pooler_output)
        toxic_logits = self.toxic_classifier(pooler_output)

        all_logits = torch.cat((topic_logits, toxic_logits), dim=1)

        loss = None
        if labels is not None:
            loss_topic = self.loss_topic_fct(topic_logits, labels[:, :5])
            loss_toxic = self.loss_toxic_fct(toxic_logits, labels[:, 5:])

            loss = loss_topic + loss_toxic

        return {"loss": loss, "logits": all_logits}

In [ ]:
# Load the model
model = XLMRobertaForMultiLabelClassification.from_pretrained('xlm-roberta-large', num_labels=11) # use 6 labels

## Training config

In [ ]:
# Define training arguments
training_args = TrainingArguments(
        num_train_epochs=6,              # Reduced from 10 (Critical)
        warmup_ratio=0.1,                # Slightly higher warmup for stability
        learning_rate=2e-5,            # Slightly lower peak LR for "Large" model
        weight_decay=0.01,

        per_device_train_batch_size=32,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=128,

        bf16=True,                       # Native A100 precision
        tf32=True,                       # TensorFloat-32 (Fast Matrix Math)
        optim="adamw_torch_fused",            # Faster compilation

        eval_strategy="steps",
        eval_steps = 200,
        save_strategy="steps",
        save_steps = 200,
        logging_steps=50,

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        dataloader_num_workers=8,
        dataloader_pin_memory=True,
        seed=42
)

def compute_metrics_gated(p):
    logits = p.predictions
    labels = p.label_ids

    probs = 1 / (1 + np.exp(-logits))

    final_preds = (probs > 0.5).astype(int)

    idx_t2 = [0, 1, 2, 3, 4]
    idx_t3 = [5, 6, 7, 8, 9, 10]

    f1_all = f1_score(labels, final_preds, average='macro')
    f1_topics = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro')

    f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro')

    return {
        'f1_macro': f1_all,
        'f1_topics': f1_topics,
        'f1_toxic': f1_toxic
    }

In [ ]:
def get_llrd_optimizer_parameters(model, learning_rate, weight_decay, lr_decay):
    # 1. Define distinct groups
    # The "Heads" get the Peak Learning Rate
    head_names = ["topic_classifier", "toxic_classifier"]

    # Standard exceptions (no weight decay for bias/LayerNorm)
    no_decay = ["bias", "LayerNorm.weight"]

    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
            "lr": learning_rate, # Peak LR (e.g., 2e-5)
        },
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in head_names) and any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": learning_rate,
        },
    ]

    # 2. Iterate Backwards through the XLM-R Encoder Layers
    # Start from top layer (closer to head) -> go down to embeddings
    layers = [model.roberta.embeddings] + list(model.roberta.encoder.layer)
    layers.reverse() # Reverse so layer 23 is first, embedding is last

    lr = learning_rate
    for layer in layers:
        lr *= lr_decay # Decay the LR (e.g., * 0.95)

        optimizer_grouped_parameters += [
            {
                "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
                "weight_decay": weight_decay,
                "lr": lr,
            },
            {
                "params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
                "lr": lr,
            },
        ]

    return optimizer_grouped_parameters

opt_parameters = get_llrd_optimizer_parameters(model, learning_rate=2e-5, weight_decay=0.01, lr_decay=0.95)
optimizer = torch.optim.AdamW(opt_parameters, lr=2e-5)

## Train

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_gated,  # Use the new metrics function
    data_collator=DataCollatorWithPadding(tokenizer),
    optimizers=(optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)]
)

In [ ]:
# Train the model
trainer.train()

## Evaluation

In [ ]:
results = trainer.evaluate()
print(f"Macro F1 score: {results['eval_f1_macro']}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import warnings

def evaluate_comprehensive(trainer, df_val, languages, task_columns, tokenizer, threshold=0.5):
    idx_t2 = [0, 1, 2, 3, 4]
    idx_t3 = [5, 6, 7, 8, 9, 10]
    results = []

    print("=" * 120)
    print(f"{'LANG':<5} | {'N':<5} | {'TOPIC F1':<8} | {'TOXIC F1':<8} | {'ALL MACRO':<9} | {'GATE RATE':<9}")
    print("=" * 120)

    # 2. Loop per Language
    for lang in languages:
        df_lang = df_val[df_val['language'] == lang]
        if len(df_lang) == 0: continue

        lang_dataset = PolarizationDataset(
            df_lang['text'].tolist(),
            df_lang[expert_columns].values.tolist(),
            tokenizer
        )

        output = trainer.predict(lang_dataset)

        probs = 1 / (1 + np.exp(-output.predictions))

        final_preds = (probs > threshold).astype(int)
        labels = output.label_ids

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            f1_topic = f1_score(labels[:, idx_t2], final_preds[:, idx_t2], average='macro', zero_division=0)
            f1_toxic = f1_score(labels[:, idx_t3], final_preds[:, idx_t3], average='macro', zero_division=0)
            f1_all = f1_score(labels, final_preds, average='macro', zero_division=0)

        results.append({
            'language': lang,
            'samples': len(df_lang),
            'f1_topic': f1_topic,
            'f1_toxic': f1_toxic,
            'f1_all': f1_all,
        })

        print(f"{lang.upper():<5} | {len(df_lang):<5} | {f1_topic:.4f}   | {f1_toxic:.4f}   | {f1_all:.4f}")

    print("=" * 120)

    return pd.DataFrame(results).sort_values('f1_all', ascending=False)


detailed_results = evaluate_comprehensive(
    trainer,
    val,
    languages,
    task_columns,
    tokenizer,
    threshold=0.5
)

print("\n--- AVERAGES ---")
print(f"Topic: {detailed_results['f1_topic'].mean():.4f}")
print(f"Toxic: {detailed_results['f1_toxic'].mean():.4f}")

## Save

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [ ]:
drive_folder = "/content/drive/MyDrive/SemEvalMultiTaskResults/Tabelas/Tabela 1/NG-XLM-RoBERTa-19L-T23-SL-PW"

if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)

In [ ]:
csv_path = os.path.join(drive_folder, "detailed_results.csv")
detailed_results.to_csv(csv_path, index=False)

In [ ]:
import time

In [ ]:
print(f"Saving model to {drive_folder}...")
trainer.save_model(drive_folder)
tokenizer.save_pretrained(drive_folder)
print("Model saved.")

print("Waiting 180 seconds for Drive background sync to finish...")
time.sleep(180)

if os.path.exists(os.path.join(drive_folder, "config.json")):
    print("SUCCESS: Config file verified. Safe to shutdown.")
    print("Shutting down runtime now, everything good")
    runtime.unassign()
else:
    print("WARNING: Something went wrong. File not found after waiting.")

print("Shutting down runtime now, no model saved :/")
runtime.unassign()

# Tabela 2

# Análise dos positivos

## Importação de Libs

In [ ]:
!unzip test_phase.zip

In [ ]:
import os
import random
import pandas as pd
import numpy as np

## Aquisição dos dados

In [ ]:
task_columns = ["polarization", "political", "racial/ethnic", "religious", "gender/sexual", "other", "stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]

task1_columns = ["polarization"]
task2_columns = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
task3_columns = ["stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]


In [ ]:
languages = [
    'arb', 'ben', 'mya', 'eng', 'hin', 'nep', 'urd', 'zho', 'amh', 'hau',
    'ori', 'fas', 'pol', 'pan', 'rus', 'spa', 'swa', 'tel', 'tur', 'ita', 'khm', 'deu'
    ]

In [ ]:
def load_and_merge_robust(lang, split_name):
    """
    Loads Subtask 1 (Mandatory).
    If Subtask 2 or 3 are missing, fills their columns with 0.
    """
    # -------------------------------------------------------
    # A. Load Subtask 1 (The Anchor)
    # -------------------------------------------------------
    path_t1 = f'/content/subtask1/{split_name}/{lang}.csv'

    # If Task 1 is missing, we really can't do anything. Skip.
    if not os.path.exists(path_t1):
        if split_name == 'train': # Only warn for train, dev might be partial
            print(f"Skipping {lang.upper()}: No Subtask 1 file found.")
        return None

    # Load the anchor dataframe
    df = pd.read_csv(path_t1)

    # -------------------------------------------------------
    # B. Load Subtask 2 (Topic)
    # -------------------------------------------------------
    path_t2 = f'/content/subtask2/{split_name}/{lang}.csv'

    if os.path.exists(path_t2):
        df_t2 = pd.read_csv(path_t2)
        # LEFT MERGE: Keep all rows from Task 1. Match IDs from Task 2.
        df = df.merge(df_t2, on=['id', 'text'], how='left')
    else:
        # File missing? Fill columns with 0
        print(f"  -> {lang.upper()} missing Subtask 2. Filling columns with 0.")
        for col in task2_columns:
            df[col] = 0

    # -------------------------------------------------------
    # C. Load Subtask 3 (Critical Step for your missing files)
    # -------------------------------------------------------
    path_t3 = f'/content/subtask3/{split_name}/{lang}.csv'

    if os.path.exists(path_t3):
        df_t3 = pd.read_csv(path_t3)
        # LEFT MERGE again
        df = df.merge(df_t3, on=['id', 'text'], how='left')
    else:
        # This will trigger for ITA, MYA, POL, RUS
        print(f"  -> {lang.upper()} missing Subtask 3. Filling columns with 0.")
        for col in task3_columns:
            df[col] = 0

    # -------------------------------------------------------
    # D. Final Cleanup
    # -------------------------------------------------------
    # The merge might create NaNs if IDs didn't match perfectly. Fill them.
    df = df.fillna(0)

    # Tag the language
    df['language'] = lang

    return df

In [ ]:
df_train = None
df_val = None
train_frames = []
val_frames = []
for lang in languages:
    # Load Train
    df_train_lang = load_and_merge_robust(lang, 'train')
    #df_train_lang.sample(frac=0.5)
    if df_train_lang is not None:
        train_frames.append(df_train_lang)

    # Load Dev
    df_val_lang = load_and_merge_robust(lang, 'dev')
    #df_val_lang.sample(frac=0.5)
    if df_val_lang is not None:
        val_frames.append(df_val_lang)

In [ ]:
train = pd.concat(train_frames, ignore_index=True)
val = pd.concat(val_frames, ignore_index=True)

In [ ]:
train['text'] = train['text'].astype(str)
val['text'] = val['text'].astype(str)

train = train[train['text'].str.strip().str.len() > 0]
val = val[val['text'].str.strip().str.len() > 0]

print(f"Cleaned Train Shape: {train.shape}")
print(f"Cleaned Val Shape: {val.shape}")

In [ ]:
path_ita_t3 = '/content/subtask2/dev/ita.csv'
if os.path.exists(path_ita_t3):
    df_raw_ita = pd.read_csv(path_ita_t3)
    print("Soma bruta no arquivo original ITA (Task 3):")
    print(df_raw_ita.drop(columns=['id', 'text'], errors='ignore').sum())
else:
    print("O arquivo ITA da Task 3 não foi encontrado no caminho especificado.")

## Análise da distribuição de train e val para cada label

In [ ]:
df_dist = pd.DataFrame(columns=['language'] + task_columns)
new_row = []
for lang in languages:
  train_temp = train[train['language'] == lang]
  val_temp = val[val['language'] == lang]
  new_row = {'language' : lang}
  for label in task_columns:
    total_samples = len(train_temp[label]) + len(val_temp[label])
    positives = (train_temp[label] == 1).sum() + (val_temp[label] == 1).sum()
    new_row[label] = positives / total_samples if total_samples > 0 else 0
  df_dist.loc[len(df_dist)] = new_row

In [ ]:
display(df_dist)

In [ ]:
df_dist.to_csv('df_dist.csv', index=False)

# Tabela 3

# Análise da variância Baseline x Ensemble

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd

def compute_metrics(df, model_name):
    tasks = {
        'Task 1': 'f1_polar',
        'Task 2': 'f1_topic',
        'Task 3': 'f1_toxic'
    }

    # Línguas que NÃO têm labels positivos na validação e devem ser ignoradas
    # (Baseado na sua Tabela 3 do df_dist)
    invalid_langs_per_task = {
        'Task 1': [],
        'Task 2': [],
        'Task 3': ['mya', 'pol', 'rus', 'ita']
    }

    records = []
    for task_name, col in tasks.items():
        # Remove as línguas estruturalmente inválidas para a Task atual
        invalid_langs = invalid_langs_per_task[task_name]
        df_filtered = df[~df['language'].isin(invalid_langs)].copy()

        # Pega as notas limpas (sem as línguas inválidas e sem NaN)
        scores = df_filtered[col].dropna()

        if len(scores) > 0:
            mean = scores.mean()
            std = scores.std(ddof=0)
            cv = std / mean if mean > 0 else 0
            min_val = scores.min()
            max_val = scores.max()
        else:
            mean = std = cv = min_val = max_val = 0

        records.append({
            'Subtask': task_name,
            'System': model_name,
            'Mean': mean,
            'Std_Dev': std,
            'CV': cv,
            'Min': min_val,
            'Max': max_val
        })

    return pd.DataFrame(records)

In [ ]:
data = [
    ["arb",0.58,0.826850,0.50,0.638630,0.30,0.596835],
    ["ben",0.78,0.852444,0.60,0.400000,0.10,0.249711],
    ["mya",0.52,0.901563,0.15,0.523946,0.00,0.000000],
    ["eng",0.52,0.822858,0.25,0.359321,0.20,0.500598],
    ["hin",0.90,0.829390,0.70,0.789091,0.30,0.749563],
    ["nep",0.76,0.909991,0.70,0.785061,0.15,0.545275],
    ["urd",0.74,0.789354,0.70,0.789587,0.45,0.801101],
    ["zho",0.32,0.919777,0.40,0.661586,0.25,0.443076],
    ["amh",0.38,0.779435,0.40,0.472200,0.15,0.494986],
    ["hau",0.20,0.893395,0.15,0.452322,0.20,0.213421],
    ["ori",0.58,0.837600,0.45,0.677660,0.15,0.276287],
    ["fas",0.66,0.859227,0.45,0.582601,0.20,0.379050],
    ["pol",0.90,0.827891,0.30,0.629781,0.00,0.000000],
    ["pan",0.76,0.879808,0.35,0.435835,0.25,0.593933],
    ["rus",0.32,0.829645,0.35,0.674983,0.00,0.000000],
    ["spa",0.64,0.744318,0.55,0.654344,0.20,0.457290],
    ["swa",0.52,0.821280,0.15,0.417728,0.30,0.566825],
    ["tel",0.86,0.898188,0.35,0.491240,0.10,0.435066],
    ["tur",0.76,0.895455,0.65,0.699145,0.25,0.570395],
]

columns = ["language","gate_th","f1_gate","topic_th","f1_topic","toxic_th","f1_toxic"]

df_ensemble = pd.DataFrame(data, columns=columns)

In [ ]:
df_baseline = pd.read_csv('/content/detailed_results.csv')

In [ ]:
df_ensemble.rename(columns={'f1_gate': 'f1_polar'}, inplace=True)

In [ ]:
metrics_baseline = compute_metrics(df_baseline, 'Baseline (22L Multi-task)')
metrics_ensemble = compute_metrics(df_ensemble, 'Proposed (Specialist Ensemble)')

df_results = pd.concat([metrics_baseline, metrics_ensemble]).sort_values(by=['Subtask', 'System'])

print("=== RESULTADOS DA DISPERSÃO ===\n")
print(df_results.to_string(index=False))

# Tabela 4

# Análise da média dos modelos ensemble x resultado baseline

## Unzip

In [ ]:
!unzip test_phase.zip

## Imports

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.49.0

In [ ]:
import os
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import numpy as np
import re
import html
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DebertaV2Model,
    DebertaV2PreTrainedModel,
    XLMRobertaModel,
    XLMRobertaPreTrainedModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    TrainerCallback
)
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')

## Dataset prep

In [ ]:
task_columns = ["polarization", "political", "racial/ethnic", "religious", "gender/sexual", "other", "stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]

task1_columns = ["polarization"]
task2_columns = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
task3_columns = ["stereotype","vilification","dehumanization","extreme_language","lack_of_empathy","invalidation"]


In [ ]:
# languages = [
#     'arb', 'ben', 'mya', 'eng', 'hin', 'nep', 'urd', 'zho', 'amh', 'hau',
#     'ori', 'fas', 'pol', 'pan', 'rus', 'spa', 'swa', 'tel', 'tur'
#     ]

languages = [
     'arb', 'ben', 'mya', 'eng', 'hin', 'nep', 'urd', 'zho', 'amh', 'hau', 'deu',
     'ori', 'fas', 'pol', 'pan', 'rus', 'spa', 'swa', 'tel', 'tur', 'ita', 'khm',
     ]

In [ ]:
def load_and_merge_robust(lang, split_name):
    """
    Loads Subtask 1 (Mandatory).
    If Subtask 2 or 3 are missing, fills their columns with 0.
    """
    # -------------------------------------------------------
    # A. Load Subtask 1 (The Anchor)
    # -------------------------------------------------------
    path_t1 = f'/content/subtask1/{split_name}/{lang}.csv'

    # If Task 1 is missing, we really can't do anything. Skip.
    if not os.path.exists(path_t1):
        if split_name == 'train': # Only warn for train, dev might be partial
            print(f"Skipping {lang.upper()}: No Subtask 1 file found.")
        return None

    # Load the anchor dataframe
    df = pd.read_csv(path_t1)

    # -------------------------------------------------------
    # B. Load Subtask 2 (Topic)
    # -------------------------------------------------------
    path_t2 = f'/content/subtask2/{split_name}/{lang}.csv'

    if os.path.exists(path_t2):
        df_t2 = pd.read_csv(path_t2)
        # LEFT MERGE: Keep all rows from Task 1. Match IDs from Task 2.
        df = df.merge(df_t2, on=['id', 'text'], how='left')
    else:
        # File missing? Fill columns with 0
        print(f"  -> {lang.upper()} missing Subtask 2. Filling columns with 0.")
        for col in task2_columns:
            df[col] = 0

    # -------------------------------------------------------
    # C. Load Subtask 3 (Critical Step for your missing files)
    # -------------------------------------------------------
    path_t3 = f'/content/subtask3/{split_name}/{lang}.csv'

    if os.path.exists(path_t3):
        df_t3 = pd.read_csv(path_t3)
        # LEFT MERGE again
        df = df.merge(df_t3, on=['id', 'text'], how='left')
    else:
        # This will trigger for ITA, MYA, POL, RUS
        print(f"  -> {lang.upper()} missing Subtask 3. Filling columns with 0.")
        for col in task3_columns:
            df[col] = 0

    # -------------------------------------------------------
    # D. Final Cleanup
    # -------------------------------------------------------
    # The merge might create NaNs if IDs didn't match perfectly. Fill them.
    df = df.fillna(0)

    # Tag the language
    df['language'] = lang

    return df

In [ ]:
df_val = None
df_test = None
val_frames = []
test_frames = []
for lang in languages:
    # Load Train
    df_val_lang = load_and_merge_robust(lang, 'dev')
    if df_val_lang is not None:
        val_frames.append(df_val_lang)

    # Load Dev
    df_test_lang = load_and_merge_robust(lang, 'test')
    if df_test_lang is not None:
        test_frames.append(df_test_lang)

In [ ]:
val = pd.concat(val_frames, ignore_index=True)
test = pd.concat(test_frames, ignore_index=True)

In [ ]:
val['text'] = val['text'].astype(str)
test['text'] = test['text'].astype(str)

val = val[val['text'].str.strip().str.len() > 0]
test = test[test['text'].str.strip().str.len() > 0]

print(f"Cleaned val Shape: {val.shape}")
print(f"Cleaned test Shape: {test.shape}")

## Ensemble specialists

In [ ]:
from transformers import AutoTokenizer

# Carregamento dos tokenizers específicos
xlmr_tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-large')
mdeb_tokenizer = AutoTokenizer.from_pretrained('microsoft/mdeberta-v3-base') # ou o modelo específico que você baixou

class EnsembleDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, xlmr_tk, mdeb_tk, max_length=128):
        self.texts = texts
        self.labels = labels
        self.xlmr_tk = xlmr_tk
        self.mdeb_tk = mdeb_tk
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        # Tokenização para XLM-R (Modelos T12, T13, Paranoid)
        xlmr_enc = self.xlmr_tk(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        # Tokenização para mDeBERTa (Modelos novos de Toxicidade)
        mdeb_enc = self.mdeb_tk(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'xlmr_ids': xlmr_enc['input_ids'].squeeze(0),
            'xlmr_mask': xlmr_enc['attention_mask'].squeeze(0),
            'mdeb_ids': mdeb_enc['input_ids'].squeeze(0),
            'mdeb_mask': mdeb_enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float)
        }

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-large')

In [ ]:
# No bloco de instanciação dos datasets:
val_dataset = EnsembleDataset(
    val['text'].tolist(),
    val[task_columns].values.tolist(),
    xlmr_tokenizer, # Passando XLM-R
    mdeb_tokenizer, # Passando mDeBERTa
    max_length=128
)

test_dataset = EnsembleDataset(
    test['text'].tolist(),
    test[task_columns].values.tolist(),
    xlmr_tokenizer,
    mdeb_tokenizer,
    max_length=128
)

### Model Configs (7 classes)

In [ ]:
try:
    _ = POS_WEIGHT
    _ = weights_for_pos
except NameError:
    POS_WEIGHT = 1.0
    weights_for_pos = [1.0] * 11

### TASK 1

#### Non-Gated SL XLMR PWC_OS_V3

In [ ]:
class XLMR_PWC_OS_V3(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)

        # 1. Gate (Polarization)
        self.gate_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1)
        )

        # 2. Topic Head
        self.topic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 5)
        )

        # 3. Toxic Head
        self.toxic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 6)
        )

        weights_tensor = torch.tensor([1.0] * 11, dtype=torch.float)

        self.loss_gate_fct = nn.BCEWithLogitsLoss()

        self.loss_topic_fct = nn.BCEWithLogitsLoss(pos_weight=weights_tensor[0:5])
        self.loss_toxic_fct = nn.BCEWithLogitsLoss(pos_weight=weights_tensor[5:])

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        # Forward Heads
        gate_logit = self.gate_classifier(pooler_output)
        topic_logits = self.topic_classifier(pooler_output)
        toxic_logits = self.toxic_classifier(pooler_output)

        all_logits = torch.cat((gate_logit, topic_logits, toxic_logits), dim=1)

        loss = None
        if labels is not None:
            loss_gate = self.loss_gate_fct(gate_logit, labels[:, 0].unsqueeze(1))
            loss_topic = self.loss_topic_fct(topic_logits, labels[:, 1:6])
            loss_toxic = self.loss_toxic_fct(toxic_logits, labels[:, 6:])
            loss = loss_gate + loss_topic + loss_toxic

        return {"loss": loss, "logits": all_logits}

#### Non-Gated SL XLMR PWC_DT_T12

In [ ]:
class XLMR_PWC_DT_T12(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)

        # 1. Gate (Polarization)
        self.gate_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1)
        )

        # 2. Topic Head
        self.topic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 5)
        )

        weights_tensor = torch.tensor([1.0]*5, dtype=torch.float)

        self.loss_gate_fct = nn.BCEWithLogitsLoss()

        # KEEPING YOUR POSITIONAL WEIGHTS (Good idea!)
        self.loss_topic_fct = nn.BCEWithLogitsLoss(pos_weight=weights_tensor)

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        # Forward Heads
        gate_logit = self.gate_classifier(pooler_output)
        topic_logits = self.topic_classifier(pooler_output)

        all_logits = torch.cat((gate_logit, topic_logits), dim=1)

        loss = None
        if labels is not None:
            # 1. Gate Loss
            loss_gate = self.loss_gate_fct(gate_logit, labels[:, 0].unsqueeze(1))

            loss_topic = self.loss_topic_fct(topic_logits, labels[:, 1:6])

            loss = loss_gate + loss_topic

        return {"loss": loss, "logits": all_logits}

#### Gated NSL XLMR PW_V2_paranoid

In [ ]:
#positional_weighting = weights_for_pos
positional_weighting = 6.0
class XLMR_PW_V2_PAR(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)

        # Gate (Polarization)
        self.gate_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1)
        )

        # Experts (Topics)
        self.expert_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size + 1, 11)
        )

        # Initialize the Loss functions here
        self.loss_gate_fct = nn.BCEWithLogitsLoss()

        # USE FOCAL LOSS FOR EXPERTS ONLY (The hard part)
        self.pos_weight = torch.tensor([positional_weighting]*11)
        self.loss_expert_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        # 1. Gate Forward
        gate_logit = self.gate_classifier(pooler_output)

        # 2. Expert Forward
        expert_input = torch.cat((pooler_output, gate_logit), dim=1)
        expert_logits = self.expert_classifier(expert_input)

        # 3. Loss Calculation
        loss = None
        if labels is not None:
            # Separate Labels
            gate_labels = labels[:, 0].unsqueeze(1)
            expert_labels = labels[:, 1:]

            # A. Calculate Gate Loss (Standard BCE is fine here)
            loss_gate = self.loss_gate_fct(gate_logit, gate_labels)

            # B. Calculate Expert Loss (Using Focal Loss to fix sparsity)
            loss_expert = self.loss_expert_fct(expert_logits, expert_labels)

            # Combine them
            loss = loss_gate + loss_expert

        logits = torch.cat((gate_logit, expert_logits), dim=1)

        return {"loss": loss, "logits": logits}

### TASK 2

#### Gated SL XLMR PW_T12

In [ ]:
#positional_weighting = weights_for_pos
positional_weighting = 2.5
class XLMR_PW_T12(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)

        # Gate (Polarization)
        self.gate_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1)
        )

        # Experts (Topics)
        self.expert_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size + 1, 5)
        )

        # Initialize the Loss functions here
        self.loss_gate_fct = nn.BCEWithLogitsLoss()

        # USE FOCAL LOSS FOR EXPERTS ONLY (The hard part)
        self.pos_weight = torch.tensor([positional_weighting]*5)
        self.loss_expert_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        # 1. Gate Forward
        gate_logit = self.gate_classifier(pooler_output)

        # 2. Expert Forward
        expert_input = torch.cat((pooler_output, gate_logit), dim=1)
        expert_logits = self.expert_classifier(expert_input)

        # 3. Loss Calculation
        loss = None
        if labels is not None:
            # Separate Labels
            gate_labels = labels[:, 0].unsqueeze(1)
            expert_labels = labels[:, 1:]

            # A. Calculate Gate Loss (Standard BCE is fine here)
            loss_gate = self.loss_gate_fct(gate_logit, gate_labels)

            # B. Calculate Expert Loss (Using Focal Loss to fix sparsity)
            loss_expert = self.loss_expert_fct(expert_logits, expert_labels)

            # Combine them
            loss = loss_gate + loss_expert

        logits = torch.cat((gate_logit, expert_logits), dim=1)

        return {"loss": loss, "logits": logits}

#### Non-Gated SL XLMR PWC_DT_T12 (classe da task 1)

#### Non-Gated SL XLMR PWC_OS_V3 (classe da task 1)

### TASK 3

#### Gated NSL XLM PW_V1

In [ ]:
#positional_weighting = weights_for_pos
positional_weighting = 6.0
class XLMR_PW_V1(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)

        # Gate (Polarization)
        self.gate_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1)
        )

        # Experts (Topics)
        self.expert_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size + 1, 11)
        )

        # Initialize the Loss functions here
        self.loss_gate_fct = nn.BCEWithLogitsLoss()

        # USE FOCAL LOSS FOR EXPERTS ONLY (The hard part)
        self.pos_weight = torch.tensor([positional_weighting]*11)
        self.loss_expert_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        # 1. Gate Forward
        gate_logit = self.gate_classifier(pooler_output)

        # 2. Expert Forward
        expert_input = torch.cat((pooler_output, gate_logit), dim=1)
        expert_logits = self.expert_classifier(expert_input)

        # 3. Loss Calculation
        loss = None
        if labels is not None:
            # Separate Labels
            gate_labels = labels[:, 0].unsqueeze(1)
            expert_labels = labels[:, 1:]

            # A. Calculate Gate Loss (Standard BCE is fine here)
            loss_gate = self.loss_gate_fct(gate_logit, gate_labels)

            # B. Calculate Expert Loss (Using Focal Loss to fix sparsity)
            loss_expert = self.loss_expert_fct(expert_logits, expert_labels)

            # Combine them
            loss = loss_gate + loss_expert

        logits = torch.cat((gate_logit, expert_logits), dim=1)

        return {"loss": loss, "logits": logits}

#### Gated MDEB NSL PW_V1

In [ ]:
#positional_weighting = weights_for_pos
positional_weighting = 6.0
class MDEB_PW_V1(DebertaV2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.deberta = DebertaV2Model(config)

        # Gate (Polarization)
        self.gate_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1)
        )

        # Experts (Topics)
        self.expert_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size + 1, 11)
        )

        # Initialize the Loss functions here
        self.loss_gate_fct = nn.BCEWithLogitsLoss()

        # USE FOCAL LOSS FOR EXPERTS ONLY (The hard part)
        self.pos_weight = torch.tensor([positional_weighting]*11)
        self.loss_expert_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        # 1. Gate Forward
        gate_logit = self.gate_classifier(pooler_output)

        # 2. Expert Forward
        expert_input = torch.cat((pooler_output, gate_logit), dim=1)
        expert_logits = self.expert_classifier(expert_input)

        # 3. Loss Calculation
        loss = None
        if labels is not None:
            # Separate Labels
            gate_labels = labels[:, 0].unsqueeze(1)
            expert_labels = labels[:, 1:]

            # A. Calculate Gate Loss (Standard BCE is fine here)
            loss_gate = self.loss_gate_fct(gate_logit, gate_labels)

            # B. Calculate Expert Loss (Using Focal Loss to fix sparsity)
            loss_expert = self.loss_expert_fct(expert_logits, expert_labels)

            # Combine them
            loss = loss_gate + loss_expert

        logits = torch.cat((gate_logit, expert_logits), dim=1)

        return {"loss": loss, "logits": logits}

#### Non-Gated XLMR SL 18L_PWC_DT_T13

In [ ]:
POS_WEIGHT = [1.0] * 6
class XLMR_18L_PWC_DT_T13(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)

        # 1. Gate (Polarization)
        self.gate_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1)
        )

        # 2. Topic Head
        self.toxic_classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 6)
        )

        weights_tensor = torch.tensor(POS_WEIGHT, dtype=torch.float)

        self.loss_gate_fct = nn.BCEWithLogitsLoss()

        # KEEPING YOUR POSITIONAL WEIGHTS (Good idea!)
        self.loss_toxic_fct = nn.BCEWithLogitsLoss(pos_weight=weights_tensor)

        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooler_output = outputs[0][:, 0, :]

        # Forward Heads
        gate_logit = self.gate_classifier(pooler_output)
        toxic_logits = self.toxic_classifier(pooler_output)

        all_logits = torch.cat((gate_logit, toxic_logits), dim=1)

        loss = None
        if labels is not None:
            # 1. Gate Loss
            loss_gate = self.loss_gate_fct(gate_logit, labels[:, 0].unsqueeze(1))

            # 2. Expert Loss
            # REMOVED MASKING: We calculate loss on the WHOLE batch.
            # This allows the Positional Weight to do its job naturally.
            # Unpolarized samples (0s) help the model learn to predict 0s.
            loss_toxic = self.loss_toxic_fct(toxic_logits, labels[:, 1:7])

            loss = loss_gate + loss_toxic

        return {"loss": loss, "logits": all_logits}

### Ensemble Config

In [ ]:
MODEL_PATHS = {
    'BEST_GATE_XLM_RoBERTa_Gated_19L_PW_DT_T12_v3': '/content/drive/MyDrive/SemEvalMultiTaskResults/TASK 1 e 2/BEST_GATE_XLM_RoBERTa_Gated_19L_PW_DT_T12_v3',
    'XLM_RoBERTa_Gated_19L_PW_v2_paranoid':         '/content/drive/MyDrive/SemEvalMultiTaskResults/TASK 1, 2 e 3/XLM_RoBERTa_Gated_19L_PW_v2_paranoid/XLM_RoBERTa_Gated_19L_PW_v2',
    'XLM_RoBERTa_Gated_19L_PWC_OS_v3':              '/content/drive/MyDrive/SemEvalMultiTaskResults/TASK 1, 2 e 3/XLM_RoBERTa_Gated_19L_PWC_OS_v3',
    'XLM_RoBERTa_Gated_19L_PW_T12':                 '/content/drive/MyDrive/SemEvalMultiTaskResults/TASK 1 e 2/XLM_RoBERTa_Gated_19L_PW_T12',
    'XLM_RoBERTa_Gated_19L_PW_v1':                  '/content/drive/MyDrive/SemEvalMultiTaskResults/TASK 1, 2 e 3/XLM_RoBERTa_Gated_19L_v1',
    'BEST_GATE_XLM_RoBERTa_Gated_18L_PW_DT_T13_v1': '/content/drive/MyDrive/SemEvalMultiTaskResults/TASK 1 e 3/BEST_GATE_XLM_RoBERTa_Gated_18L_PW_DT_T13_v1',
    'MDeBERTa_Gated_19L_PW_v1': '/content/drive/MyDrive/SemEvalMultiTaskResults/TASK 1, 2 e 3/MDeBERTa_Gated_19L_PW_v1'
}

MODEL_CLASSES = {
    'BEST_GATE_XLM_RoBERTa_Gated_19L_PW_DT_T12_v3': XLMR_PWC_DT_T12,
    'XLM_RoBERTa_Gated_19L_PW_v2_paranoid':         XLMR_PW_V2_PAR,
    'XLM_RoBERTa_Gated_19L_PWC_OS_v3':              XLMR_PWC_OS_V3,
    'XLM_RoBERTa_Gated_19L_PW_T12':                 XLMR_PW_T12,
    'XLM_RoBERTa_Gated_19L_PW_v1':                  XLMR_PW_V1,
    'BEST_GATE_XLM_RoBERTa_Gated_18L_PW_DT_T13_v1': XLMR_18L_PWC_DT_T13,
    'MDeBERTa_Gated_19L_PW_v1': MDEB_PW_V1
}

In [ ]:
gate_team = [
            'BEST_GATE_XLM_RoBERTa_Gated_19L_PW_DT_T12_v3',
            'XLM_RoBERTa_Gated_19L_PW_v2_paranoid',
            'XLM_RoBERTa_Gated_19L_PWC_OS_v3'
        ]

topic_team = [
            'XLM_RoBERTa_Gated_19L_PW_T12',
            'BEST_GATE_XLM_RoBERTa_Gated_19L_PW_DT_T12_v3',
            'XLM_RoBERTa_Gated_19L_PWC_OS_v3'
        ]

toxic_team = [
            'XLM_RoBERTa_Gated_19L_PW_v1',
            'MDeBERTa_Gated_19L_PW_v1',
            'BEST_GATE_XLM_RoBERTa_Gated_18L_PW_DT_T13_v1'
        ]

In [ ]:
class TaskSpecificEnsemble(nn.Module):
    def __init__(self, paths, classes, device='cuda'):
        super().__init__()
        self.device = device
        self.models = nn.ModuleDict()
        self.excluded_langs = ['mya', 'ita', 'pol', 'rus']

        all_unique_names = set(gate_team + topic_team + toxic_team)
        print(f"--- Iniciando carregamento de {len(all_unique_names)} modelos ---")

        for name in all_unique_names:
            if name not in paths or name not in classes:
                continue

            current_model_class = classes[name]
            path = paths[name]

            try:
                print(f"📥 Carregando: {name}...")
                # .half() reduz o uso de VRAM pela metade (FP16)
                model_instance = current_model_class.from_pretrained(path).half()
                model_instance.to(self.device)
                model_instance.eval()
                self.models[name] = model_instance
            except Exception as e:
                print(f"❌ ERRO em {name}: {str(e)}")

    def forward(self, xlmr_ids, xlmr_mask, mdeb_ids, mdeb_mask, langs):
        batch_size = xlmr_ids.size(0)

        gate_accum = torch.zeros(batch_size, 1, device=self.device)
        topic_accum = torch.zeros(batch_size, 5, device=self.device)
        toxic_accum = torch.zeros(batch_size, 6, device=self.device)

        gate_count = torch.zeros(batch_size, 1, device=self.device)
        topic_count = torch.zeros(batch_size, 5, device=self.device)
        toxic_count = torch.zeros(batch_size, 6, device=self.device)

        gate_weights = {
            'BEST_GATE_XLM_RoBERTa_Gated_19L_PW_DT_T12_v3': 1.5,
            'XLM_RoBERTa_Gated_19L_PW_v2_paranoid': 0.7,
            'XLM_RoBERTa_Gated_19L_PWC_OS_v3': 0.7
        }

        with torch.no_grad():
            for name, model in self.models.items():
                # ROTEAMENTO DE TOKENIZER: mDeBERTa usa mdeb_ids, outros usam xlmr_ids
                ids = mdeb_ids if "MDeB" in name or "MDeBERTa" in name else xlmr_ids
                mask = mdeb_mask if "MDeB" in name or "MDeBERTa" in name else xlmr_mask

                # Inference em half precision
                out = model(ids, mask)
                logits = out['logits'].float() if isinstance(out, dict) else out[0].float()

                num_cols = logits.shape[1]

                # --- GATE ---
                if name in gate_team:
                    w = gate_weights.get(name, 1.0)
                    gate_accum += logits[:, 0:1] * w
                    gate_count += w

                # --- TOPIC ---
                if name in topic_team and num_cols >= 6:
                    topic_accum += logits[:, 1:6]
                    topic_count += 1.0

                # --- TOXIC ---
                if name in toxic_team:
                    t_logits = logits[:, 6:12] if num_cols == 12 else logits[:, 0:6]
                    if "18L" in name:
                        m = torch.tensor([l not in self.excluded_langs for l in langs],
                                         device=self.device).float().unsqueeze(1)
                        toxic_accum += t_logits * m
                        toxic_count += m
                    else:
                        toxic_accum += t_logits
                        toxic_count += 1.0

        return torch.cat([
            gate_accum / gate_count.clamp(min=0.001),
            topic_accum / topic_count.clamp(min=0.001),
            toxic_accum / toxic_count.clamp(min=0.001)
        ], dim=1)

In [ ]:
def get_ensemble_predictions(dataset, df_metadata, ensemble_model, batch_size=32):
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_logits = []
    lang_list = df_metadata['language'].tolist()

    for i, batch in enumerate(tqdm(dataloader, desc="Predicting")):
        # Mover os 4 tensores para GPU
        x_ids = batch['xlmr_ids'].to(ensemble_model.device)
        x_mask = batch['xlmr_mask'].to(ensemble_model.device)
        m_ids = batch['mdeb_ids'].to(ensemble_model.device)
        m_mask = batch['mdeb_mask'].to(ensemble_model.device)

        start_idx = i * batch_size
        end_idx = start_idx + x_ids.size(0)
        batch_langs = lang_list[start_idx:end_idx]

        # Chamada do forward com assinatura atualizada
        logits = ensemble_model(x_ids, x_mask, m_ids, m_mask, batch_langs)
        all_logits.append(logits.cpu().numpy())

    return np.concatenate(all_logits, axis=0)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ensemble = TaskSpecificEnsemble(MODEL_PATHS, MODEL_CLASSES, device=device)
val_logits = get_ensemble_predictions(val_dataset, val, ensemble)

### Médias de cada modelo do ensemble

In [ ]:
def evaluate_models_comparison(ensemble_model, df_val, languages, task_columns, xlmr_tokenizer, mdeb_tokenizer, threshold=0.5):
    """
    Retorna um DataFrame com: model, avg Task 1, avg Task 2, avg Task 3
    A média (avg) é a média aritmética dos F1-Macros de cada idioma.
    """
    comparison_data = []

    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]
    idx_t3 = [6, 7, 8, 9, 10, 11]

    excluded_langs_t3 = ["rus", "pol", "ita", "mya"]

    for model_name, model in ensemble_model.models.items():
        print(f"\n🔍 Inspecionando modelo: {model_name}...")

        lang_f1s_t1 = []
        lang_f1s_t2 = []
        lang_f1s_t3 = []

        # Variáveis para log
        debug_t3_langs = []

        for lang in languages:
            df_lang = df_val[df_val['language'] == lang]
            if len(df_lang) == 0:
                continue # Idioma não está no dataset de validação

            lang_dataset = EnsembleDataset(
                df_lang['text'].tolist(),
                df_lang[task_columns].values.tolist(),
                xlmr_tokenizer,
                mdeb_tokenizer,
                max_length=128
            )

            dataloader = DataLoader(lang_dataset, batch_size=32, shuffle=False)
            model.eval()
            all_logits = []
            with torch.no_grad():
              for batch in dataloader:
                  is_mdeb = "MDeB" in model_name or "mDeBERTa" in model_name
                  ids = batch['mdeb_ids'].to(ensemble_model.device) if is_mdeb else batch['xlmr_ids'].to(ensemble_model.device)
                  mask = batch['mdeb_mask'].to(ensemble_model.device) if is_mdeb else batch['xlmr_mask'].to(ensemble_model.device)

                  out = model(ids, mask)
                  logits = out['logits'].float() if isinstance(out, dict) else out[0].float()
                  all_logits.append(logits.cpu().numpy())

            logits = np.concatenate(all_logits, axis=0)
            probs = 1 / (1 + np.exp(-logits))
            preds = (probs > threshold).astype(int)
            labels = df_lang[task_columns].values

            num_cols = logits.shape[1]

            # --- Task 1 ---
            f1_t1 = f1_score(labels[:, idx_t1], preds[:, 0:1], average='macro', zero_division=0)
            lang_f1s_t1.append(f1_t1)

            # --- Task 2 ---
            if num_cols >= 6 and "T13" not in model_name:
                f1_t2 = f1_score(labels[:, idx_t2], preds[:, 1:6], average='macro', zero_division=0)
                lang_f1s_t2.append(f1_t2)

            # --- Task 3 ---
            if lang not in excluded_langs_t3:
                f1_t3 = None
                if num_cols == 12:
                    f1_t3 = f1_score(labels[:, idx_t3], preds[:, 6:12], average='macro', zero_division=0)
                elif "T13" in model_name or "18L" in model_name:
                    f1_t3 = f1_score(labels[:, idx_t3], preds[:, 1:7], average='macro', zero_division=0)

                if f1_t3 is not None:
                    lang_f1s_t3.append(f1_t3)
                    debug_t3_langs.append((lang.upper(), f1_t3))

        # === PRINT DE DEBUG PARA TASK 3 ===
        if debug_t3_langs:
            print("   -> F1 Task 3 por idioma:")
            for l, f1 in debug_t3_langs:
                print(f"      {l}: {f1:.4f}")
            print(f"   -> MÉDIA T3 REGISTRADA: {np.mean(lang_f1s_t3):.4f}")
        elif (num_cols == 12 or "T13" in model_name or "18L" in model_name):
            print("   -> ALERTA: Nenhum idioma válido encontrado para avaliar a Task 3 neste modelo.")

        comparison_data.append({
            'model': model_name,
            'avg Task 1': np.mean(lang_f1s_t1) if lang_f1s_t1 else 0,
            'avg Task 2': np.mean(lang_f1s_t2) if lang_f1s_t2 else 0,
            'avg Task 3': np.mean(lang_f1s_t3) if lang_f1s_t3 else 0
        })

    return pd.DataFrame(comparison_data)

df_comparison = evaluate_models_comparison(ensemble, val, languages, task_columns, xlmr_tokenizer, mdeb_tokenizer)
display(df_comparison.sort_values(by='avg Task 1', ascending=False))

### Resultado do Ensemble sem th optimization

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

val_probs = 1 / (1 + np.exp(-val_logits))
val_labels = np.array(val[task_columns].values.tolist())

print("\nAvaliando Ensemble no dataset de Validação (Threshold Fixo = 0.5)...")

results_dict = {}
languages_val = val['language'].unique()

# Índices das tarefas
idx_gate = 0
idx_topics = slice(1, 6)
idx_toxic = slice(6, 12)

# Lista de exclusão para a Task 3 (Toxic)
excluded_langs_t3 = ["rus", "pol", "ita", "mya"]

# Threshold fixo padrão para não otimizado
DEFAULT_TH = 0.5

print(f"{'LANG':<5} | {'GATE F1':<8} | {'TOPIC F1':<8} | {'TOXIC F1':<8}")
print("-" * 45)

for lang in languages_val:
    mask = (val['language'] == lang).values
    if not np.any(mask): continue

    p = val_probs[mask]
    l = val_labels[mask]

    # --- 1. GATE (Threshold Fixo) ---
    pred_g = (p[:, idx_gate] > DEFAULT_TH).astype(int)
    f1_g = f1_score(l[:, idx_gate], pred_g, average='macro', zero_division=0)

    # Máscara binária do Gate para cascata
    gate_passed = (p[:, idx_gate] > DEFAULT_TH).reshape(-1, 1)

    # --- 2. ESPECIALISTAS (Threshold Fixo condicionado ao Gate) ---

    # TOPIC
    f_p_top = (gate_passed & (p[:, idx_topics] > DEFAULT_TH)).astype(int)
    f1_top = f1_score(l[:, idx_topics], f_p_top, average='macro', zero_division=0)

    # TOXIC (Aplicando a lógica de exclusão)
    if lang in excluded_langs_t3:
        f1_tox = np.nan # Usa NaN para não afetar a média global do Pandas no final
    else:
        f_p_tox = (gate_passed & (p[:, idx_toxic] > DEFAULT_TH)).astype(int)
        f1_tox = f1_score(l[:, idx_toxic], f_p_tox, average='macro', zero_division=0)

    # Salva os resultados
    results_dict[lang] = {
        'gate_th': DEFAULT_TH, 'f1_gate': f1_g,
        'topic_th': DEFAULT_TH, 'f1_topic': f1_top,
        'toxic_th': DEFAULT_TH, 'f1_toxic': f1_tox
    }

    # Formatação condicional para o log na tela
    f1_tox_str = f"{f1_tox:.4f}" if not pd.isna(f1_tox) else "N/A"

    print(f"{lang.upper():<5} | "
          f"{f1_g:.4f}   | "
          f"{f1_top:.4f}   | "
          f"{f1_tox_str:<8}")

print("-" * 45)
print("Avaliação concluída. Gerando DataFrame final...")

# 2. Criação do DataFrame de Resultados
df_results = pd.DataFrame.from_dict(results_dict, orient='index').reset_index()
df_results.rename(columns={'index': 'language'}, inplace=True)

# 3. Cálculo das Médias Globais (Ponderadas por língua)
# O Pandas já ignora os NaNs por padrão ao chamar o .mean()
avg_f1_gate = df_results['f1_gate'].mean()
avg_f1_topic = df_results['f1_topic'].mean()
avg_f1_toxic = df_results['f1_toxic'].mean()

print(f"\n--- MÉDIAS GLOBAIS DO ENSEMBLE (Macro F1 por Língua - Sem Otimização) ---")
print(f"F1 Gate Absolute:  {avg_f1_gate:.4f}")
print(f"F1 Topic Expert:   {avg_f1_topic:.4f} (Calculado sobre polarizados)")
print(f"F1 Toxic Expert:   {avg_f1_toxic:.4f} (Excluindo pol, rus, mya, ita)")

## Ensemble Joint Multitask

In [ ]:
# Fix the dataset class by inheriting from torch.utils.data.Dataset
class PolarizationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length # Store max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding=False, max_length=self.max_length, return_tensors='pt')

        # Ensure consistent tensor conversion for all items
        item = {key: encoding[key].squeeze() for key in encoding.keys()}
        # CHANGE THIS LINE: Use torch.float instead of torch.long for multi-label classification
        item['labels'] = torch.tensor(label, dtype=torch.float)
        return item

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-large', num_labels=12)

# Create train and Test dataset for multilabel
val_dataset = PolarizationDataset(val['text'].tolist(), val[task_columns].values.tolist(), tokenizer)

### Model config

In [ ]:
from torch.utils.data import DataLoader, Dataset
class PolarizationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        # CORREÇÃO 1: padding='max_length' para evitar tamanhos incompatíveis no batch
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        item = {key: encoding[key].squeeze() for key in encoding.keys()}
        item['labels'] = torch.tensor(label, dtype=torch.float)
        return item

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-large')

# Create Datasets
val_dataset = PolarizationDataset(val['text'].tolist(), val[task_columns].values.tolist(), tokenizer)

### Ensemble Config

In [ ]:
MODEL_PATHS = {
    'NG-XLM-RoBERTa-22L-T123-SL': '/content/drive/MyDrive/SemEvalMultiTaskResults/Tabelas/Tabela 1/NG-XLM-RoBERTa-22L-T123-SL',
    'NG-XLM-RoBERTa-22L-T123-SL_rerun': '/content/drive/MyDrive/SemEvalMultiTaskResults/Tabelas/Tabela 4/NG-XLM-RoBERTa-22L-T123-SL_rerun',
    'NG-XLM-RoBERTa-22L-T123-SL_rerun_v2':  '/content/drive/MyDrive/SemEvalMultiTaskResults/Tabelas/Tabela 4/NG-XLM-RoBERTa-22L-T123-SL_rerun_2',
}

In [ ]:
class TaskSpecificEnsemble(nn.Module):
    def __init__(self, paths, model_config, device='cuda'):
        super().__init__()
        self.device = device
        self.models = nn.ModuleDict()

        print(f"--- Iniciando carregamento de {len(paths)} modelos ---")

        for name in paths:
            current_model_class = model_config
            path = paths[name]
            try:
                print(f"📥 Carregando: {name}...")
                model_instance = current_model_class.from_pretrained(path).half()
                model_instance.to(self.device)
                model_instance.eval()
                self.models[name] = model_instance
            except Exception as e:
                print(f"❌ ERRO em {name}: {str(e)}")

    def forward(self, input_ids, attention_mask, langs=None):
        batch_size = input_ids.size(0)
        total_logits = torch.zeros(batch_size, 12, device=self.device)
        num_models = len(self.models)

        with torch.no_grad():
            for name, model in self.models.items():
                out = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = out['logits'].float() if isinstance(out, dict) else out[0].float()
                total_logits += logits

        return total_logits / num_models

In [ ]:
def get_ensemble_predictions(dataset, df_metadata, ensemble_model, batch_size=32):
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_logits = []

    # Previne erro se a coluna language não existir no momento
    lang_list = df_metadata['language'].tolist() if 'language' in df_metadata.columns else [None]*len(dataset)

    for i, batch in enumerate(tqdm(dataloader, desc="Predicting")):
        # CHAVES CORRIGIDAS AQUI:
        ids = batch['input_ids'].to(ensemble_model.device)
        mask = batch['attention_mask'].to(ensemble_model.device)

        start_idx = i * batch_size
        end_idx = start_idx + ids.size(0)
        batch_langs = lang_list[start_idx:end_idx]

        logits = ensemble_model(input_ids=ids, attention_mask=mask, langs=batch_langs)
        all_logits.append(logits.cpu().numpy())

    return np.concatenate(all_logits, axis=0)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ensemble = TaskSpecificEnsemble(MODEL_PATHS, XLMRobertaForMultiLabelClassification, device=device)

In [ ]:
val_logits = get_ensemble_predictions(val_dataset, val, ensemble)

### Médias dos 3 modelos

In [ ]:
def evaluate_models_comparison(ensemble_model, df_val, languages, task_columns, tokenizer, threshold=0.5):
    comparison_data = []
    idx_t1 = [0]
    idx_t2 = [1, 2, 3, 4, 5]
    idx_t3 = [6, 7, 8, 9, 10, 11]

    excluded_langs_t3 = ["rus", "pol", "ita", "mya"]

    for model_name, model in ensemble_model.models.items():
        print(f"\n🔍 Inspecionando modelo: {model_name}...")

        lang_f1s_t1, lang_f1s_t2, lang_f1s_t3 = [], [], []

        for lang in languages:
            df_lang = df_val[df_val['language'] == lang]
            if len(df_lang) == 0: continue

            # CORREÇÃO 4: Classe Dataset correta
            lang_dataset = PolarizationDataset(
                df_lang['text'].tolist(),
                df_lang[task_columns].values.tolist(),
                tokenizer,
                max_length=128
            )

            dataloader = DataLoader(lang_dataset, batch_size=32, shuffle=False)
            model.eval()
            all_logits = []

            with torch.no_grad():
                for batch in dataloader:
                    # CORREÇÃO 5: Chaves corretas
                    ids = batch['input_ids'].to(ensemble_model.device)
                    mask = batch['attention_mask'].to(ensemble_model.device)

                    out = model(input_ids=ids, attention_mask=mask)
                    logits = out['logits'].float() if isinstance(out, dict) else out[0].float()
                    all_logits.append(logits.cpu().numpy())

            logits = np.concatenate(all_logits, axis=0)
            probs = 1 / (1 + np.exp(-logits))
            preds = (probs > threshold).astype(int)
            labels = df_lang[task_columns].values

            # ... Lógica de F1 que você escreveu continua a mesma ...
            f1_t1 = f1_score(labels[:, idx_t1], preds[:, 0:1], average='macro', zero_division=0)
            lang_f1s_t1.append(f1_t1)

            f1_t2 = f1_score(labels[:, idx_t2], preds[:, 1:6], average='macro', zero_division=0)
            lang_f1s_t2.append(f1_t2)

            if lang not in excluded_langs_t3:
                f1_t3 = f1_score(labels[:, idx_t3], preds[:, 6:12], average='macro', zero_division=0)
                lang_f1s_t3.append(f1_t3)

        comparison_data.append({
            'model': model_name,
            'avg Task 1': np.mean(lang_f1s_t1) if lang_f1s_t1 else 0,
            'avg Task 2': np.mean(lang_f1s_t2) if lang_f1s_t2 else 0,
            'avg Task 3': np.mean(lang_f1s_t3) if lang_f1s_t3 else 0
        })

    return pd.DataFrame(comparison_data)

languages = val['language'].unique()
df_comparison = evaluate_models_comparison(ensemble, val, languages, task_columns, tokenizer)
display(df_comparison.sort_values(by='avg Task 1', ascending=False))

### Ensemble dos 3 modelos joint

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

val_probs = 1 / (1 + np.exp(-val_logits))
val_labels = np.array(val[task_columns].values.tolist())

print("\nAvaliando Ensemble no dataset de Validação (Threshold Fixo = 0.5)...")

results_dict = {}
languages_val = val['language'].unique()

# Índices das tarefas
idx_gate = 0
idx_topics = slice(1, 6)
idx_toxic = slice(6, 12)

# Lista de exclusão para a Task 3 (Toxic)
excluded_langs_t3 = ["rus", "pol", "ita", "mya"]

# Threshold fixo padrão para não otimizado
DEFAULT_TH = 0.5

print(f"{'LANG':<5} | {'GATE F1':<8} | {'TOPIC F1':<8} | {'TOXIC F1':<8}")
print("-" * 45)

for lang in languages_val:
    mask = (val['language'] == lang).values
    if not np.any(mask): continue

    p = val_probs[mask]
    l = val_labels[mask]

    # --- 1. GATE (Threshold Fixo) ---
    pred_g = (p[:, idx_gate] > DEFAULT_TH).astype(int)
    f1_g = f1_score(l[:, idx_gate], pred_g, average='macro', zero_division=0)

    # Máscara binária do Gate para cascata
    gate_passed = (p[:, idx_gate] > DEFAULT_TH).reshape(-1, 1)

    # --- 2. ESPECIALISTAS (Threshold Fixo condicionado ao Gate) ---

    # TOPIC
    f_p_top = (gate_passed & (p[:, idx_topics] > DEFAULT_TH)).astype(int)
    f1_top = f1_score(l[:, idx_topics], f_p_top, average='macro', zero_division=0)

    # TOXIC (Aplicando a lógica de exclusão)
    if lang in excluded_langs_t3:
        f1_tox = np.nan # Usa NaN para não afetar a média global do Pandas no final
    else:
        f_p_tox = (gate_passed & (p[:, idx_toxic] > DEFAULT_TH)).astype(int)
        f1_tox = f1_score(l[:, idx_toxic], f_p_tox, average='macro', zero_division=0)

    # Salva os resultados
    results_dict[lang] = {
        'gate_th': DEFAULT_TH, 'f1_gate': f1_g,
        'topic_th': DEFAULT_TH, 'f1_topic': f1_top,
        'toxic_th': DEFAULT_TH, 'f1_toxic': f1_tox
    }

    # Formatação condicional para o log na tela
    f1_tox_str = f"{f1_tox:.4f}" if not pd.isna(f1_tox) else "N/A"

    print(f"{lang.upper():<5} | "
          f"{f1_g:.4f}   | "
          f"{f1_top:.4f}   | "
          f"{f1_tox_str:<8}")

print("-" * 45)
print("Avaliação concluída. Gerando DataFrame final...")

# 2. Criação do DataFrame de Resultados
df_results = pd.DataFrame.from_dict(results_dict, orient='index').reset_index()
df_results.rename(columns={'index': 'language'}, inplace=True)

# 3. Cálculo das Médias Globais (Ponderadas por língua)
# O Pandas já ignora os NaNs por padrão ao chamar o .mean()
avg_f1_gate = df_results['f1_gate'].mean()
avg_f1_topic = df_results['f1_topic'].mean()
avg_f1_toxic = df_results['f1_toxic'].mean()

print(f"\n--- MÉDIAS GLOBAIS DO ENSEMBLE (Macro F1 por Língua - Sem Otimização) ---")
print(f"F1 Gate Absolute:  {avg_f1_gate:.4f}")
print(f"F1 Topic Expert:   {avg_f1_topic:.4f} (Calculado sobre polarizados)")
print(f"F1 Toxic Expert:   {avg_f1_toxic:.4f} (Excluindo pol, rus, mya, ita)")